# Afetlerde Hızlı Risk Profillemesi: Sentetik Veri Üretimi + Model Karşılaştırması (tek parça)

Bu defter, makalede raporlanan deney hattının tamamını tek seferde çalıştırır:

1. **Veri üretimi:** Claude API ile İstanbul'un 39 ilçesine dağıtılmış, ihtiyaç etiketli, Türkçe + İngilizce simüle sosyal medya paylaşımları üretir (varsayılan 2.000 kayıt, Opus 5.5).
2. **Boru hattı:** Düz / lemmalanmış / gövdelenmiş metin → 7 dil temsil modeli (+ isteğe bağlı ek model) → ihtiyaç listesiyle kosinüs benzerliği → ilçe bazlı ihtiyaç tespiti ve doğruluk.
3. **BDM karşılaştırması ve çıktılar:** Claude (ve anahtar varsa GPT / Gemini) ile sıfır atış değerlendirme; tablolar (CSV/XLSX/MD), şekiller (PNG), haritalar (HTML/PNG), özet rapor ve zip.
4. **Eşit koşullar paketi:** model ayarları tablosu (parametre, boyut, eğitim verisi, havuzlama, ince ayar), ham/ince ayarlı eş çiftler (BERT-raw, ALBERT-paraphrase), temel çizgiler (TF-IDF, rastgele, çoğunluk), Türkçe modeller, CPU'da ölçülen gecikme, eşleştirilmiş anlamlılık testleri (McNemar + bootstrap, Holm) ve üretilen her kaydın bağımsız bir modelle (Sonnet 5.5) etiket denetimi.

**Çalışma zamanı önerisi:** Runtime > Change runtime type > **GPU: L4 veya A100**, **High-RAM açık**. GPU yalnızca transformer modellerini hızlandırır; asıl ihtiyaç RAM'dir (FastText cc.en.300.bin tek başına ~7 GB, Word2Vec ~3,6 GB, GloVe ~2,6 GB). 12 GB'lık standart çalışma zamanında FastText/GloVe yüklemesi çökebilir.

**Gizli anahtar:** Sol menüdeki anahtar simgesinden `ANTHROPIC_API_KEY` tanımlı ve bu defter için erişim açık olmalı (isteğe bağlı: `OPENAI_API_KEY`, `GOOGLE_API_KEY`).

**Çıktılar:** `MyDrive/NLP_Disaster/` altında `data/` (üretilen veri), `runs/<etiket>/` (tablolar, şekiller, haritalar, günlükler, `ozet.md`) ve `sonuclar_<etiket>.zip` (otomatik indirilir).

**Maliyet denetimi:** Yeni modellerde düşünme (thinking) varsayılan olarak açıktır ve düşünme token'ları çıktı fiyatından faturalanır. Defter bunu `THINKING_MODE="minimal"` ve `GEN_EFFORT="low"` ile sınırlar (Opus 5.5'te düşünme kapatılamaz, yalnız azaltılır), her 10 çağrıda ortalama token ve tahmini maliyeti yazar ve `BUDGET_GUARD_USD` aşılırsa yeni çağrı yapmaz. Tahmini üretim maliyeti: Opus 5.5 ~12-15 $, Sonnet 5.5 ~4-6 $ (2.000 kayıt).

**Hızlı deneme:** Ayarlar hücresinde `SMOKE_TEST = True` yaparsanız API'ye ve büyük modellere dokunmadan, sahte veri ve sahte gömmelerle tüm akış 2-3 dakikada çalışır (kod akışını denemek için).

**Not:** Tüm ayarlar yalnızca bir sonraki hücrede değiştirilir; diğer hücrelere dokunmanız gerekmez. Üretim parçaları Drive'a kaydedilir; defter yarıda kesilirse yeniden çalıştırıldığında kaldığı yerden devam eder (aynı ayarlarla yeniden ücret ödenmez).

In [ ]:
# =============================================================================
# 0. AYARLAR  (değiştirilecek tek yer)
# =============================================================================
SMOKE_TEST = False            # True: API ve büyük modeller olmadan sahte veri/modelle 2-3 dakikalık deneme

# --- Klasörler -----------------------------------------------------------------
DRIVE_FOLDER_NAME = "NLP_Disaster"   # Google Drive > MyDrive altındaki klasör (yoksa oluşturulur)
RUN_TAG = None                       # None: otomatik tarih-saat etiketi; örn. "deneme1"

# --- Veri üretimi (Claude API) -------------------------------------------------
N_MESSAGES = 2000                    # hedef kayıt sayısı (39 ilçeye eşit dağıtılır)
GEN_MODEL = "claude-opus-5-5"        # üretim modeli. Tahmini üretim maliyeti (2.000 kayıt, THINKING_MODE="minimal"):
                                     #   claude-opus-5-5 ~12-15 $ (düşünme kapatılamaz, effort ile sınırlanır) | claude-sonnet-5-5 ~4-6 $ | claude-haiku-5-5 ~0,5 $
GEN_EFFORT = "low"                   # düşünme derinliği: "low" en ucuz/hızlı; None: API varsayılanı (Opus 5.5'te "medium", çok daha pahalı)
THINKING_MODE = "minimal"            # "minimal": Sonnet 5.5 -> between_tools, Haiku 5.5 -> disabled, Opus/Fable -> yalnız effort; "default": adaptif düşünme açık (pahalı)
GEN_RECORDS_PER_CALL = 20            # bir API çağrısında üretilen kayıt sayısı (üretim imzasına girer; değiştirirseniz önceki parçalar kullanılmaz)
GEN_WORKERS = 4                      # eş zamanlı çağrı sayısı (hız limiti aşılırsa otomatik bekler)
GEN_MAX_TOKENS_PER_RECORD = 700      # kayıt başına çıktı üst sınırı (düşünme payı dâhil; kullanılmayan token ücretlendirilmez)
BUDGET_GUARD_USD = 30.0              # bu oturumda tahmini toplam API maliyeti bunu aşarsa yeni çağrı yapılmaz (None: kapalı)
GEN_TEMPERATURE = None               # None: API varsayılanı. Güncel SDK (>=1.15) ve yeni modeller sıcaklık parametresi almaz; eski SDK'da sayı verilirse kullanılır
GEN_OVERSAMPLE = 1.15                # tekrar/kalite elemesi için fazladan üretim payı
IMPLICIT_RATIO = 0.30                # ihtiyacı adıyla söylemeyip durumdan çıkarılmasını gerektiren paylaşım oranı
MIN_PER_NEED = 12                    # her ihtiyacın etiket olarak en az kaç kez geçeceği
DEDUP_COSINE_THRESHOLD = 0.92        # bu benzerliğin üzerindeki paylaşımlar tekrar sayılır ve atılır
FORCE_REGENERATE = False             # True: Drive'daki önceki üretim parçalarını yok sayıp yeniden üret (ücret!)
RUN_LABEL_QC = True                  # üretilen her kaydı BAĞIMSIZ bir modelle denetle (etiketler metinde var mı, fazladan ihtiyaç var mı)
QC_MODEL = "claude-sonnet-5-5"       # kalite kontrol modeli (üretim modelinden farklı olmalı); tahmini ~1,5-2 $
QC_EFFORT = "low"
QC_EXCLUDE_FAILED = False            # True: denetimden geçemeyen kayıtları veri setinden çıkar (ilçe kotaları bozulabilir); False: yalnız işaretle

# --- İhtiyaç listesi -----------------------------------------------------------
FIX_NEED_TRANSLATIONS = True         # True: hatalı Google Translate karşılıkları düzeltilir ("voluntarily" vb.)
                                     # False: makine çevirisi karşılıkları düzeltilmeden kullanılır (64 benzersiz etiket)
PREPROCESS_NEEDS_LIKE_MESSAGES = True  # True: lemma/gövde deneylerinde ihtiyaç adları da aynı işlemden geçer
                                       # False: ihtiyaç adları ham bırakılır

# --- Dil temsil modelleri ---------------------------------------------------------
MODELS_TO_RUN = ["BERT", "ALBERT", "RoBERTa", "GloVe", "Word2Vec", "USE", "FastText"]  # makaledeki 7 model
EXTRA_MODELS = ["SBERT-mpnet", "BERT-raw", "ALBERT-paraphrase"]  # ek modeller: güncel SBERT + ham/ince ayarlı eş çiftler; kapatmak için []
TURKISH_MODELS = ["SBERT-çokdilli (TR)", "BERTurk-STS (TR)"]      # Türkçe metin + Türkçe ihtiyaç listesi (yalnız düz metin); kapatmak için []
BASELINES = ["Temel: TF-IDF", "Temel: Rastgele", "Temel: Çoğunluk"]  # şans düzeyi ve sözcüksel temel çizgi
TEXT_TYPES = ["plain", "lemma", "stem"]   # düz / lemmalanmış / gövdelenmiş
LATENCY_ON_CPU = True                # tek mesaj gecikmesi tüm yerel modeller için CPU'da ölçülür (eşit koşul); toplu GPU süresi ayrıca verilir
SIGNIFICANCE_TESTS = True            # en iyi modele karşı eşleştirilmiş McNemar + bootstrap fark testleri (Holm düzeltmeli)
GLOVE_VARIANT = "840B"               # "840B" (makaledeki Common Crawl 840B) veya "wiki300" (küçük, hızlı yedek)
CACHE_MODELS_TO_DRIVE = False        # True: indirilen büyük modelleri Drive'a da kopyala (Drive kotası ~15 GB ister)
MODEL_DIR_CANDIDATES = {             # kullanıcının Drive'ında zaten bulunan model dosyaları (ilk bulunan kullanılır)
    "Word2Vec": ["MyDrive/Colab Notebooks/NLP Modeller/Word2Vec Files/word2vec-google-news-300.bin",
                 "MyDrive/NLP_Disaster/models/word2vec-google-news-300.bin"],
    "GloVe":    ["MyDrive/NLP_Disaster/models/glove.840B.300d.kv",
                 "MyDrive/Colab Notebooks/NLP Modeller/GloVe File/glove.840B.300d.txt",
                 "MyDrive/Colab Notebooks/NLP Modeller/GloVe File/glove.840B.300d-001.txt"],
    "FastText": ["MyDrive/Colab Notebooks/NLP Modeller/FastText Files/cc.en.300.bin",
                 "MyDrive/NLP_Disaster/models/cc.en.300.bin"],
    "USE":      ["MyDrive/Colab Notebooks/NLP Modeller/USE Files",
                 "MyDrive/NLP_Disaster/models/USE"],
}
BOOTSTRAP_N = 1000                   # güven aralığı için yeniden örnekleme sayısı

# --- BDM (büyük dil modeli) değerlendirmesi, Tablo 5 ----------------------------------
LLM_EVAL_MODELS = [                  # (sağlayıcı, model). openai/google satırları anahtar yoksa atlanır
    ("anthropic", "claude-opus-5-5"),
    ("anthropic", "claude-sonnet-5-5"),
    ("anthropic", "claude-haiku-5-5"),
    ("openai", "gpt-5-mini"),
    ("google", "gemini-3.8-flash"),   # gemini-2.5-flash yeni kullanıcılara kapatıldı (API 404)
]
LLM_EVAL_MESSAGES_PER_CALL = 10      # 1: her paylaşım ayrı istekte gönderilir (daha yavaş)
LLM_EVAL_EFFORT = "low"              # BDM değerlendirmesinde düşünme derinliği ("low": en düşük ve en ucuz düzey)
LLM_EVAL_WORKERS = 4                 # Anthropic için eş zamanlı istek sayısı
LLM_EVAL_WORKERS_BY_PROVIDER = {"openai": 2, "google": 2}   # diğer sağlayıcılar (düşük kademe hız sınırları için düşük tutuldu)
LLM_EVAL_RPM_BY_PROVIDER = {"openai": None, "google": None}  # dakikadaki istek üst sınırı; None: uyarlanabilir (429 geldikçe bekler).
                                                             # Düşük kademe (yeni hesap) ise örn. "openai": 8 yazın.
LLM_MIN_COVERAGE = 0.98              # bir BDM'nin mesajların en az bu oranı için yanıtı yoksa (kota/limit) Tablo 5'e ALINMAZ,
                                     # "eksik" olarak ayrı dosyaya yazılır; yeniden çalıştırınca önbellekten kaldığı yerden devam eder
LLM_EVAL_MAX_MESSAGES = None         # None: tüm mesajlar; örn. 500: rastgele alt küme (hızlı/ucuz)
RUN_LATENCY_BENCHMARK = True         # tüm BDM'ler için aynı paylaşımlarla, sıralı çağrılarla gecikme ve maliyet ölçümü (Tablo 6b)
BENCH_BATCH_REQUESTS = 15            # toplu kip: 15 istek x 10 paylaşım (ms/paylaşım ve USD/1.000 paylaşım)
BENCH_SINGLE_REQUESTS = 15           # tekil kip: 15 istek x 1 paylaşım (tek paylaşım yanıt süresi)
BENCH_FORCE = False                  # True: önceki ölçümleri yok sayıp yeniden ölç

# --- Genel ---------------------------------------------------------------------------
SEED = 42
AUTO_DOWNLOAD_ZIP = True             # bitince sonuç zip'ini tarayıcıya indir
PRICES_USD_PER_MTOK = {              # maliyet tahmini için (giriş, çıkış, önbellek okuma, önbellek yazma)
    "claude-opus-5-5":   (4.0, 20.0, 0.20, 5.0),
    "claude-sonnet-5-5": (2.0, 10.0, 0.10, 2.5),
    "claude-haiku-5-5":  (0.10, 0.50, 0.01, 0.125),
    "gpt-5-mini":        (0.25, 2.00, 0.025, 0.0),    # OpenAI liste fiyatı (Ekim 2026); önbellek yazma ücretsiz
    "gemini-3.8-flash":  (0.75, 3.75, 0.075, 0.0),    # Google liste fiyatı (Ekim 2026; 31 Aralık 2026'ya kadar geçerli; ai.google.dev/gemini-api/docs/pricing)
}

In [ ]:
# =============================================================================
# 1. KURULUM: paketler, Drive, klasörler, günlük, çalışma zamanı bilgisi
# =============================================================================
import os, sys, re, io, json, time, math, gc, random, hashlib, shutil, subprocess, warnings, datetime, zipfile, urllib.request
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings("ignore")

def pip_install(pkgs, upgrade=False):
    """Sessiz pip kurulumu (hata olursa akışı durdurmaz, uyarı verir). Colab'ın hazır numpy/pandas/torch sürümlerine dokunmaz."""
    if os.environ.get("NLP_SKIP_PIP"):
        return
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + (["--upgrade"] if upgrade else []) + pkgs
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("UYARI: pip kurulumu başarısız:", pkgs, "\n", r.stderr[-800:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

pip_install(["anthropic>=1.0"], upgrade=True)          # output_config (yapılandırılmış JSON) için güncel SDK gerekir
pip_install(["folium", "nltk", "inflect", "openpyxl", "tqdm", "psutil"])
if not SMOKE_TEST:
    pip_install(["sentence-transformers", "gensim>=4.3.3", "tensorflow_hub"])

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from tqdm.auto import tqdm
import nltk
os.environ.setdefault("NLTK_ALLOW_PROXIED_URLOPEN", "1")   # vekil sunucu arkasındaki ortamlarda NLTK indirmesine izin ver
for res in ["wordnet", "omw-1.4", "stopwords", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(res, quiet=True)
    except Exception as e:
        print("NLTK indirme uyarısı:", res, e)

# --- Rastgelelik ---------------------------------------------------------------
random.seed(SEED); np.random.seed(SEED)
RNG = random.Random(SEED)
NP_RNG = np.random.default_rng(SEED)

# --- Drive ve klasörler --------------------------------------------------------
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive"
    BASE_DIR = os.path.join(DRIVE_ROOT, "MyDrive", DRIVE_FOLDER_NAME)
    LOCAL_SCRATCH = "/content/nlp_scratch"
else:
    DRIVE_ROOT = os.path.abspath("./drive_local")           # Colab dışı deneme için yerel klasör
    BASE_DIR = os.path.join(DRIVE_ROOT, "MyDrive", DRIVE_FOLDER_NAME)
    LOCAL_SCRATCH = os.path.abspath("./nlp_scratch")

if RUN_TAG is None:
    RUN_TAG = datetime.datetime.now().strftime("%Y%m%d_%H%M") + ("_smoke" if SMOKE_TEST else "")
RUN_DIR   = os.path.join(BASE_DIR, "runs", RUN_TAG)
DATA_DIR  = os.path.join(BASE_DIR, "data")
MODEL_DIR = os.path.join(BASE_DIR, "models")
DIRS = {
    "data": DATA_DIR, "models": MODEL_DIR, "run": RUN_DIR,
    "results": os.path.join(RUN_DIR, "results"), "figures": os.path.join(RUN_DIR, "figures"),
    "maps": os.path.join(RUN_DIR, "maps"), "logs": os.path.join(RUN_DIR, "logs"),
    "preds": os.path.join(RUN_DIR, "results", "tahminler"), "llm_cache": os.path.join(DATA_DIR, "bdm_cache"),
    "scratch": LOCAL_SCRATCH,
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

# --- Günlük --------------------------------------------------------------------
LOG_PATH = os.path.join(DIRS["logs"], "log.txt")
def log(*args):
    msg = " ".join(str(a) for a in args)
    stamp = datetime.datetime.now().strftime("%H:%M:%S")
    line = f"[{stamp}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# --- Çalışma zamanı bilgisi -----------------------------------------------------
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "yok"
except Exception:
    torch = None; DEVICE = "cpu"; GPU_NAME = "yok (torch bulunamadı)"
try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / 1e9
except Exception:
    RAM_GB = float("nan")
log(f"Colab: {IN_COLAB} | GPU: {GPU_NAME} | RAM: {RAM_GB:.0f} GB | Python {sys.version.split()[0]}")
log(f"Çıktı klasörü: {RUN_DIR}")
if not SMOKE_TEST and RAM_GB < 25:
    log("UYARI: RAM 25 GB'ın altında. FastText (cc.en.300.bin, ~7 GB) ve GloVe 840B yüklemesi çökebilir. "
        "Runtime > Change runtime type > High-RAM önerilir.")

# --- API anahtarları -----------------------------------------------------------
def get_secret(name):
    val = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            val = None
    return val or os.environ.get(name)

ANTHROPIC_API_KEY = get_secret("ANTHROPIC_API_KEY")
OPENAI_API_KEY    = get_secret("OPENAI_API_KEY")
GOOGLE_API_KEY    = get_secret("GOOGLE_API_KEY")
if not SMOKE_TEST and not ANTHROPIC_API_KEY:
    raise RuntimeError("ANTHROPIC_API_KEY bulunamadı. Colab'da sol menü > Anahtar simgesi > "
                       "ANTHROPIC_API_KEY ekleyip 'Notebook access' açın.")
log("Anahtarlar: Anthropic=%s | OpenAI=%s | Google=%s" % tuple("var" if k else "yok" for k in (ANTHROPIC_API_KEY, OPENAI_API_KEY, GOOGLE_API_KEY)))

# --- Yardımcılar ------------------------------------------------------------------
def tr_lower(s):
    """Türkçe'ye uygun küçük harf (İ→i, I→ı)."""
    return str(s).replace("İ", "i").replace("I", "ı").lower()

def fmt_tr(x, nd=2):
    """Ondalık virgüllü sayı biçimi (makale Türkçe)."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    return f"{x:.{nd}f}".replace(".", ",")

def save_table(df, name, index=False, float_nd=3):
    """Tabloyu CSV + XLSX + Markdown olarak kaydeder."""
    p = os.path.join(DIRS["results"], name)
    df.to_csv(p + ".csv", index=index, encoding="utf-8-sig")
    try:
        df.to_excel(p + ".xlsx", index=index)
    except Exception as e:
        log("XLSX kaydedilemedi:", name, e)
    with open(p + ".md", "w", encoding="utf-8") as f:
        f.write(df_to_markdown(df, index=index, float_nd=float_nd))
    return p

def df_to_markdown(df, index=False, float_nd=3):
    d = df.reset_index() if index else df
    cols = list(d.columns)
    def cell(v):
        if isinstance(v, float):
            return fmt_tr(v, float_nd)
        return str(v)
    lines = ["| " + " | ".join(str(c) for c in cols) + " |", "|" + "---|" * len(cols)]
    for _, row in d.iterrows():
        lines.append("| " + " | ".join(cell(row[c]) for c in cols) + " |")
    return "\n".join(lines) + "\n"

plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
TR_DECIMAL = FuncFormatter(lambda v, pos: f"{v:.2f}".replace(".", ","))
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000", "#999999"]  # Okabe-Ito

CONFIG = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, list, dict, tuple, type(None)))
          and k not in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY")}
with open(os.path.join(DIRS["logs"], "config.json"), "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=2, default=str)
log("Kurulum tamam.")

In [ ]:
# =============================================================================
# 2. İHTİYAÇ LİSTESİ VE İLÇELER
# =============================================================================
# 66 maddelik ihtiyaç listesi (sıra korunmuştur) ve makine çevirisiyle (Google Translate) elde edilen İngilizce karşılıklar.
NEEDS_TR = [
    "içme suyu", "bebek maması", "biberon", "ıslak mendil", "peçete", "gıda", "fener", "elektrikli ısıtıcı", "soba", "odun",
    "pil", "battaniye", "pilli radyo", "diş fırçası", "diş macunu", "hijyenik ped", "bebek bezi", "sabun", "yedek giysi", "mont",
    "maske", "nakit para", "konserve", "sıcak yemek", "ilaç", "ağrı kesici", "antibiyotik", "soğuk algınlığı ilacı", "ateş düşürücü", "yarabandı",
    "bandaj", "alerji ilaçları", "astım ilacı", "insülin", "seyyar tuvalet", "çadır", "uyku tulumu", "arama kurtarma ekibi", "sağlık ekibi", "psikolojik destek ekibi",
    "güvenlik güçleri", "jenaratör", "powerbank", "glutesiz yiyecek", "tuzsuz-şekersiz yiyecek", "gönüllü", "kuru gıda", "ekmek", "ilk yardım kiti", "kap kacak",
    "engelli ekipmanı", "kan", "aydınlatıcı", "hayvan maması", "kalp ilacı", "konteynır", "temizlik ürünü", "hırdavat", "çocuk oyun alanı", "oksijen cihazı",
    "elektrik saha ekibi", "tercüman", "kan toplama merkezi", "can yeleği", "mutfak eşyası", "el feneri",
]
NEEDS_EN_ORIG = [
    "drinking water", "baby food", "bottle", "baby wipes", "napkin", "food", "torch", "electric heater", "stove", "wood",
    "battery", "blanket", "battery radio", "toothbrush", "toothpaste", "sanitary pad", "diaper", "soap", "spare clothes", "coat",
    "mask", "cash money", "canned", "hot meal", "medicine", "painkiller", "antibiotic", "cold medicine", "antipyretic", "bandage",
    "bandage", "allergy medications", "asthma medicine", "insulin", "portable toilet", "tent", "sleeping bag", "search and rescue team", "medical team", "psychological support team",
    "security forces", "generator", "power bank", "gluten free food", "salt-free-sugar-free food", "voluntarily", "dry food", "bread", "first aid kit", "utensils",
    "disabled equipment", "blood", "enlightening", "pet food", "heart medicine", "container", "cleaning product", "hardware", "children's playground", "oxygen device",
    "electrical field team", "interpreter", "blood collection center", "life jacket", "kitchenware", "torch",
]
assert len(NEEDS_TR) == len(NEEDS_EN_ORIG) == 66

# Makine çevirisindeki hatalı / yinelenen karşılıklar için düzeltmeler (FIX_NEED_TRANSLATIONS=True iken uygulanır)
NEEDS_EN_FIX = {
    "bebek maması": "baby formula", "biberon": "baby bottle", "ıslak mendil": "wet wipes", "fener": "lantern",
    "el feneri": "flashlight", "soba": "heating stove", "odun": "firewood", "pil": "batteries", "pilli radyo": "battery-powered radio",
    "nakit para": "cash", "konserve": "canned food", "ateş düşürücü": "fever reducer", "yarabandı": "adhesive bandage",
    "alerji ilaçları": "allergy medication", "tuzsuz-şekersiz yiyecek": "salt-free and sugar-free food", "gönüllü": "volunteers",
    "kap kacak": "pots and pans", "engelli ekipmanı": "equipment for disabled people", "aydınlatıcı": "lighting equipment",
    "konteynır": "container housing", "hırdavat": "hardware tools", "oksijen cihazı": "oxygen concentrator",
}
NEED_TABLE = pd.DataFrame({"tr": NEEDS_TR, "en_orig": NEEDS_EN_ORIG})
NEED_TABLE["en_fixed"] = [NEEDS_EN_FIX.get(tr, en) for tr, en in zip(NEEDS_TR, NEEDS_EN_ORIG)]
NEED_TABLE["en"] = NEED_TABLE["en_fixed"] if FIX_NEED_TRANSLATIONS else NEED_TABLE["en_orig"]
TR2EN = dict(zip(NEED_TABLE["tr"], NEED_TABLE["en"]))
EN_LABELS = list(dict.fromkeys(NEED_TABLE["en"]))          # benzersiz İngilizce etiketler (model çıktı uzayı)
EN2TR = {}
for tr, en in zip(NEED_TABLE["tr"], NEED_TABLE["en"]):
    EN2TR.setdefault(en, tr)                                 # yinelenen etikette ilk Türkçe karşılık
log(f"İhtiyaç listesi: {len(NEEDS_TR)} Türkçe madde, {len(EN_LABELS)} benzersiz İngilizce etiket "
    f"(çeviri düzeltme: {'açık' if FIX_NEED_TRANSLATIONS else 'kapalı'})")
if FIX_NEED_TRANSLATIONS:
    changed = NEED_TABLE[NEED_TABLE.en_orig != NEED_TABLE.en_fixed][["tr", "en_orig", "en_fixed"]]
    log("Düzeltilen çeviriler:\n" + changed.to_string(index=False))
NEED_TABLE.to_csv(os.path.join(DIRS["results"], "ihtiyac_listesi.csv"), index=False, encoding="utf-8-sig")

# Yazım farkları: etiketlerde ve üretim çıktılarında görülebilen farklı yazımlar liste adına eşlenir
NEED_ALIASES_TR = {"jeneratör": "jenaratör", "jenerator": "jenaratör", "glutensiz yiyecek": "glutesiz yiyecek", "glütensiz yiyecek": "glutesiz yiyecek",
                   "power bank": "powerbank", "konteyner": "konteynır", "yara bandı": "yarabandı", "hijyenik bağ": "hijyenik ped"}
NEED_DISPLAY_TR = {"jenaratör": "jeneratör", "glutesiz yiyecek": "glutensiz yiyecek"}   # tablo/rapor gösterimi için doğru yazım
def canon_need_tr(x):
    """Türkçe ihtiyaç adını liste biçimine getirir (küçük harf, boşluk, yazım eşlemesi); listede yoksa None."""
    if not isinstance(x, str):
        return None
    k = " ".join(tr_lower(x).split())
    k = NEED_ALIASES_TR.get(k, k)
    return k if k in TR2EN else None

def get_english_need(tr_need):
    return TR2EN.get(tr_lower(tr_need).strip(), None)

# --- 39 ilçe ve merkez koordinatları (Nominatim ile belirlenmiştir) ------------
DISTRICTS = [
    "Adalar", "Arnavutköy", "Ataşehir", "Avcılar", "Bağcılar", "Bahçelievler", "Bakırköy", "Başakşehir", "Bayrampaşa", "Beşiktaş",
    "Beykoz", "Beylikdüzü", "Beyoğlu", "Büyükçekmece", "Çatalca", "Çekmeköy", "Esenler", "Esenyurt", "Eyüpsultan", "Fatih",
    "Gaziosmanpaşa", "Güngören", "Kadıköy", "Kağıthane", "Kartal", "Küçükçekmece", "Maltepe", "Pendik", "Sancaktepe", "Sarıyer",
    "Silivri", "Sultanbeyli", "Sultangazi", "Şile", "Şişli", "Tuzla", "Ümraniye", "Üsküdar", "Zeytinburnu",
]
DISTRICT_COORDS = {
    "Adalar": (40.8741659, 29.1293251), "Arnavutköy": (41.184471, 28.7412446), "Ataşehir": (40.9929379, 29.1135187),
    "Avcılar": (40.9799389, 28.7216689), "Bağcılar": (41.034547, 28.8567558), "Bahçelievler": (41.0030495, 28.8657701),
    "Bakırköy": (40.9782585, 28.8744461), "Başakşehir": (41.1075794, 28.7950711), "Bayrampaşa": (41.0345549, 28.9118417),
    "Beşiktaş": (41.0428465, 29.0075283), "Beykoz": (41.1343001, 29.0920378), "Beylikdüzü": (41.0037541, 28.6372583),
    "Beyoğlu": (41.0284233, 28.9736808), "Büyükçekmece": (41.021654, 28.579757), "Çatalca": (41.1436804, 28.4605154),
    "Çekmeköy": (41.0351579, 29.1739149), "Esenler": (41.0376175, 28.8824519), "Esenyurt": (41.0342862, 28.6801113),
    "Eyüpsultan": (41.0478358, 28.9327383), "Fatih": (41.0192846, 28.9479296), "Gaziosmanpaşa": (41.0578305, 28.9122452),
    "Güngören": (41.0252832, 28.8726498), "Kadıköy": (40.9912955, 29.0245631), "Kağıthane": (41.0796544, 28.9731198),
    "Kartal": (40.88858, 29.1856536), "Küçükçekmece": (40.9918737, 28.7711956), "Maltepe": (40.9247539, 29.1310782),
    "Pendik": (40.8768715, 29.2349672), "Sancaktepe": (40.9905196, 29.2288624), "Sarıyer": (41.1685803, 29.0572623),
    "Silivri": (41.0742476, 28.2481709), "Sultanbeyli": (40.9670242, 29.2671314), "Sultangazi": (41.1043344, 28.8614367),
    "Şile": (41.1744067, 29.6125216), "Şişli": (41.0637891, 28.9831642), "Tuzla": (40.8161732, 29.3034194),
    "Ümraniye": (41.0256362, 29.0963049), "Üsküdar": (41.0265498, 29.0151321), "Zeytinburnu": (40.9898653, 28.9037467),
}
assert len(DISTRICTS) == 39 and all(d in DISTRICT_COORDS for d in DISTRICTS)
DISTRICTS_LOWER = {tr_lower(d): d for d in DISTRICTS}

def detect_districts(text):
    """Alt dize eşleşmesiyle ilçe adı tespiti; Türkçe büyük/küçük harf kurallarına uygun ve harf duyarsız."""
    t = tr_lower(text)
    return [d for d in DISTRICTS if tr_lower(d) in t]

# --- İlçe sınırları (statik haritalar için) -------------------------------------------------------
# Kaynak: geoBoundaries (gbOpen, TUR ADM2; OpenStreetMap türevi, ODbL / CC BY 4.0, kullanım koşulu: kaynak belirtme).
# Dosya bir kez indirilip Drive'da saklanır (data/geo); indirme başarısız olursa haritalar nokta işaretleriyle çizilir.
import unicodedata
GEO_DIR = os.path.join(DATA_DIR, "geo"); os.makedirs(GEO_DIR, exist_ok=True)
GEO_PATH = os.path.join(GEO_DIR, "istanbul_ilceler.geojson")
GEO_SOURCE_NOTE = "geoBoundaries gbOpen TUR ADM2 (OpenStreetMap verisi, ODbL / CC BY 4.0)"

def _geo_norm(s):
    s = str(s).replace("İ", "i").replace("I", "ı").lower()
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    return s.replace("ı", "i").replace("ş", "s").replace("ğ", "g").replace("ç", "c").replace("ö", "o").replace("ü", "u").replace(" ", "")

_GEO_ALIASES = {"eyup": "Eyüpsultan", "eyupsultan": "Eyüpsultan"}
_IST_BBOX = (27.95, 40.70, 29.95, 41.62)   # lon_min, lat_min, lon_max, lat_max

def _geom_lon_lat(geom):
    """Poligonların tüm (lon, lat) noktalarını düz liste olarak verir."""
    pts = []
    if geom["type"] == "Polygon":
        for ring in geom["coordinates"]:
            pts.extend(ring)
    elif geom["type"] == "MultiPolygon":
        for poly in geom["coordinates"]:
            for ring in poly:
                pts.extend(ring)
    return pts

def fetch_istanbul_geojson(force=False):
    """İstanbul'un 39 ilçesinin sınırlarını geoBoundaries'ten alır ve GEO_PATH'e yazar; yol ya da None döner."""
    if os.path.exists(GEO_PATH) and not force:
        return GEO_PATH
    try:
        import requests
        meta = requests.get("https://www.geoboundaries.org/api/current/gbOpen/TUR/ADM2/", timeout=60).json()
        url = meta.get("simplifiedGeometryGeoJSON") or meta.get("gjDownloadURL")
        gj = requests.get(url, timeout=300).json()
        wanted = {_geo_norm(d): d for d in DISTRICTS}
        feats = []
        for f in gj["features"]:
            name = f["properties"].get("shapeName", "")
            d = wanted.get(_geo_norm(name)) or _GEO_ALIASES.get(_geo_norm(name))
            if not d:
                continue
            pts = _geom_lon_lat(f["geometry"])
            if not pts:
                continue
            lon = sum(p[0] for p in pts) / len(pts); lat = sum(p[1] for p in pts) / len(pts)
            if not (_IST_BBOX[0] <= lon <= _IST_BBOX[2] and _IST_BBOX[1] <= lat <= _IST_BBOX[3]):
                continue   # aynı adlı başka ilçe (İstanbul dışı)
            feats.append({"type": "Feature", "properties": {"ilce": d, "shapeName": name, "shapeID": f["properties"].get("shapeID", "")}, "geometry": f["geometry"]})
        found = {f["properties"]["ilce"] for f in feats}
        missing = [d for d in DISTRICTS if d not in found]
        if len(found) < 30:
            log(f"İlçe sınırları: yalnızca {len(found)} ilçe eşleşti, sınırlar kullanılmayacak. Eksik: {missing}")
            return None
        out = {"type": "FeatureCollection", "name": "istanbul_ilceler", "source": GEO_SOURCE_NOTE,
               "boundary_year": meta.get("boundaryYearRepresented"), "license": meta.get("licenseDetail"), "features": feats}
        with open(GEO_PATH, "w", encoding="utf-8") as fh:
            json.dump(out, fh, ensure_ascii=False)
        log(f"İlçe sınırları indirildi: {len(found)} ilçe ({GEO_SOURCE_NOTE})" + (f"; eksik: {missing}" if missing else ""))
        return GEO_PATH
    except Exception as e:
        log("İlçe sınırları indirilemedi (haritalar nokta işaretleriyle çizilecek):", repr(e))
        return None

GEOJSON_PATH = fetch_istanbul_geojson()

In [ ]:
# =============================================================================
# 3. VERİ ÜRETİMİ: etiket-öncelikli (label-first) sentetik paylaşım üretimi
# -----------------------------------------------------------------------------
# Tasarım: Önce her kayıt için ilçe, 1-3 gerçek ihtiyaç ve üslup özellikleri (persona, kayıt dili,
# uzunluk, yazım hatası, hashtag, emoji, mahalle, sayı, açık/örtük anlatım) örneklenir ("plan").
# Model yalnızca bu plana uygun Türkçe paylaşımı ve İngilizce karşılığını yazar. Böylece etiketler
# kesin olur, ihtiyaç ve ilçe dağılımı kontrol edilir, her ilçenin "baskın ihtiyaç profili" olur.
# =============================================================================
PROMPT_VERSION = "v1"   # DİKKAT: üretim imzasına girer; değiştirmeyin (önbellekteki parçalar yeniden kullanılsın)

PERSONAS = [
    "yaşlı bir kadın", "yaşlı bir adam", "üniversite öğrencisi", "üç çocuklu bir anne", "yeni doğum yapmış bir anne",
    "apartman yöneticisi", "mahalle muhtarı", "gönüllü bir arama kurtarma eri", "enkaz altında kalmış, telefon şarjı azalan bir kişi",
    "komşusu adına yazan bir kişi", "tekerlekli sandalye kullanan engelli bir birey", "hamile bir kadın", "evcil hayvan sahibi",
    "bakkal (esnaf)", "ilkokul öğretmeni", "Türkçesi sınırlı bir mülteci", "sağlık çalışanı (hemşire)",
    "şehir dışındaki, ailesine ulaşamayan bir akraba", "cami görevlisi", "site güvenlik görevlisi", "taksi şoförü", "lise öğrencisi",
    "diyaliz hastası bir yakını olan kişi", "yurtta kalan öğrenci", "inşaat işçisi", "ev hanımı", "emekli öğretmen", "kafe çalışanı",
    "sokak hayvanlarına bakan bir gönüllü", "yabancı turist (basit Türkçe)",
]
REGISTERS = [("panik, büyük harfler ve ünlemler", 0.25), ("sakin ve bilgilendirici", 0.25), ("kısa ve telgraf gibi", 0.15),
             ("duygusal ve yalvaran", 0.15), ("argo ve kısaltmalı gündelik dil", 0.12), ("resmi ve kurumsal", 0.08)]
LENGTHS = [("kısa (6-14 kelime)", 0.35), ("orta (15-30 kelime)", 0.45), ("uzun (31-50 kelime)", 0.20)]
PRIOR_WEIGHTS = defaultdict(lambda: 1.0, {
    "arama kurtarma ekibi": 6, "içme suyu": 5, "gıda": 4, "battaniye": 4, "çadır": 4, "sağlık ekibi": 4, "sıcak yemek": 3,
    "ilaç": 3, "bebek maması": 3, "bebek bezi": 2, "jenaratör": 2, "elektrikli ısıtıcı": 2, "el feneri": 2, "powerbank": 2,
    "kuru gıda": 2, "ekmek": 2, "seyyar tuvalet": 2, "uyku tulumu": 2, "mont": 2, "ilk yardım kiti": 2, "psikolojik destek ekibi": 2,
    "insülin": 1.5, "soba": 1.5, "pil": 1.5, "maske": 1.5, "yedek giysi": 1.5, "konteynır": 1.5, "hayvan maması": 1.5,
})
GEN_SIGNATURE = hashlib.md5(json.dumps([N_MESSAGES, GEN_MODEL, SEED, PROMPT_VERSION, IMPLICIT_RATIO, FIX_NEED_TRANSLATIONS,
                                        GEN_RECORDS_PER_CALL, MIN_PER_NEED, SMOKE_TEST]).encode()).hexdigest()[:10]
PARTS_DIR = os.path.join(DATA_DIR, "gen_parts", GEN_SIGNATURE)
os.makedirs(PARTS_DIR, exist_ok=True)
log(f"Üretim imzası: {GEN_SIGNATURE} (parçalar: {PARTS_DIR})")

def weighted_choice(rng, items_with_w):
    items, ws = zip(*items_with_w)
    return rng.choices(items, weights=ws, k=1)[0]

# --- İlçe profilleri: her ilçenin 1 birincil, 1 ikincil, 1 üçüncül ihtiyacı -----------------
prior_arr = np.array([PRIOR_WEIGHTS[n] for n in NEEDS_TR], dtype=float)
primary_needs = [str(x) for x in NP_RNG.choice(NEEDS_TR, size=len(DISTRICTS), replace=False, p=prior_arr / prior_arr.sum())]
DISTRICT_PROFILE = {}
for d, p1 in zip(DISTRICTS, primary_needs):
    rest = [n for n in NEEDS_TR if n != p1]
    w = np.array([PRIOR_WEIGHTS[n] for n in rest]); w = w / w.sum()
    p2, p3 = NP_RNG.choice(rest, size=2, replace=False, p=w)
    DISTRICT_PROFILE[d] = [p1, str(p2), str(p3)]

def sample_needs(rng, district):
    p1, p2, p3 = DISTRICT_PROFILE[district]
    r = rng.random()
    if r < 0.40:   first = p1
    elif r < 0.58: first = p2
    elif r < 0.70: first = p3
    else:          first = weighted_choice(rng, [(n, PRIOR_WEIGHTS[n]) for n in NEEDS_TR])
    n_needs = weighted_choice(rng, [(1, 0.60), (2, 0.30), (3, 0.10)])
    needs = [str(first)]
    while len(needs) < n_needs:
        cand = weighted_choice(rng, [(n, PRIOR_WEIGHTS[n]) for n in NEEDS_TR if n not in needs])
        needs.append(str(cand))
    return needs

def sample_style(rng, n_needs):
    r = rng.random()
    if r < IMPLICIT_RATIO:               explicitness = "implicit"
    elif r < IMPLICIT_RATIO + 0.15 and n_needs > 1: explicitness = "mixed"
    else:                                explicitness = "explicit"
    return {
        "persona": rng.choice(PERSONAS), "register": weighted_choice(rng, REGISTERS), "length": weighted_choice(rng, LENGTHS),
        "explicitness": explicitness, "typos": rng.random() < 0.30, "no_diacritics": rng.random() < 0.20,
        "hashtag": rng.random() < 0.30, "emoji": rng.random() < 0.20, "neighbourhood": rng.random() < 0.50, "numbers": rng.random() < 0.50,
    }

def build_specs(quota_by_district, round_idx):
    """Plan: ilçe kotalarına göre kayıt özellikleri. Etiket tabanları (MIN_PER_NEED) 0. turda uygulanır."""
    rng = random.Random(SEED * 1000 + round_idx)
    specs = []
    for d in DISTRICTS:
        for _ in range(int(quota_by_district.get(d, 0))):
            needs = sample_needs(rng, d)
            specs.append({"district": d, "needs_tr": needs, **sample_style(rng, len(needs))})
    if round_idx == 0:
        counts = Counter(n for s in specs for n in s["needs_tr"])
        bg = [i for i, s in enumerate(specs) if s["needs_tr"][0] not in DISTRICT_PROFILE[s["district"]]]
        rng.shuffle(bg)
        for need in NEEDS_TR:
            deficit = MIN_PER_NEED - counts[need]
            while deficit > 0 and bg:
                i = bg.pop()
                old = specs[i]["needs_tr"][0]
                if need in specs[i]["needs_tr"]:
                    continue
                specs[i]["needs_tr"][0] = need
                counts[old] -= 1; counts[need] += 1; deficit -= 1
    rng.shuffle(specs)
    for k, s in enumerate(specs):
        s["id"] = f"r{round_idx}_{k:05d}"
        s["needs_en"] = [TR2EN[n] for n in s["needs_tr"]]
        s["round"] = round_idx
    return specs

# --- İstem metinleri -------------------------------------------------------------
NEEDS_REFERENCE = "\n".join(f"- {tr} ({en})" for tr, en in zip(NEED_TABLE["tr"], NEED_TABLE["en"]))
SYSTEM_PROMPT_GEN = f"""You generate realistic simulated social media posts for an academic research dataset on rapid needs detection after a major earthquake in Istanbul, Türkiye (first 72 hours, winter). Posts are written by affected residents, relatives, neighbours and volunteers.

You receive a list of post specifications. For EACH specification write:
- "message_tr": the post in natural Turkish, exactly as a real person would post on X/Twitter in that situation (max 280 characters).
- "message_en": the same post as the same person would have written it in English (not a stiff translation; keep the meaning, the district name, the informality, roughly the same length; max 280 characters).

Hard rules:
1. Mention the assigned district exactly once, spelled exactly as given (e.g. "Kadıköy", "Şişli"), optionally with a Turkish suffix (Kadıköy'de, Şişli'den, Fatihte). Never mention any other Istanbul district from this list: {", ".join(DISTRICTS)}. Avoid neighbourhood or street names that contain another district's name (e.g. do not write "Kartaltepe" or "Maltepe Mahallesi" outside those districts). Keep the district name correctly spelled in BOTH languages, even when "no_diacritics" is true.
2. The post must convey ALL of the assigned needs ("needs_tr"/"needs_en") and no other item from the needs list below.
   - "explicit": name the item(s) plainly (everyday wording and synonyms are fine, e.g. "mama" for bebek maması).
   - "implicit": do NOT name the item; describe the situation so that a human would infer it (e.g. "my grandmother is diabetic and her injections stayed under the rubble" -> insülin; "the baby has had nothing since last night" -> bebek maması).
   - "mixed": the first need explicit, the others implicit.
3. Follow the persona, register and length. Feature flags: "typos": include 1-2 realistic Turkish typos or missing letters (never in the district name). "no_diacritics": drop Turkish diacritics in some words (never in the district name). "hashtag": add 1-2 relevant hashtags (#deprem, #İstanbul, a district tag). "emoji": 1-2 emojis. "neighbourhood": add a plausible real neighbourhood, street or landmark of that district. "numbers": include concrete numbers (people, hours, floors, ages).
4. No phone numbers, no real personal names, no URLs, no @mentions. Vary sentence openings and structure; do not reuse a template; do not start most posts with "Acil".
5. Return ONLY a JSON object with the key "records": a list with one object per specification, same "id" values, same order, fields: id, district, message_tr, message_en.

Needs list (Turkish (English)); for reference only, use everyday wording in posts:
{NEEDS_REFERENCE}
"""
GEN_SCHEMA = {
    "type": "object",
    "properties": {"records": {"type": "array", "items": {
        "type": "object",
        "properties": {"id": {"type": "string"}, "district": {"type": "string"},
                       "message_tr": {"type": "string"}, "message_en": {"type": "string"}},
        "required": ["id", "district", "message_tr", "message_en"], "additionalProperties": False}}},
    "required": ["records"], "additionalProperties": False,
}
SPEC_KEYS = ["id", "district", "needs_tr", "needs_en", "explicitness", "persona", "register", "length",
             "typos", "no_diacritics", "hashtag", "emoji", "neighbourhood", "numbers"]
def build_user_prompt(specs):
    lines = [json.dumps({k: s[k] for k in SPEC_KEYS}, ensure_ascii=False) for s in specs]
    return "Specifications (one JSON object per line):\n" + "\n".join(lines) + f"\n\nWrite {len(specs)} posts now."
with open(os.path.join(DIRS["logs"], "prompt_uretim.txt"), "w", encoding="utf-8") as f:
    f.write(SYSTEM_PROMPT_GEN + "\n\n--- örnek kullanıcı istemi ---\n" + build_user_prompt(build_specs({DISTRICTS[0]: 2}, 99)))

# --- Claude istemcisi ve yapılandırılmış JSON çağrısı ---------------------------------------
import threading
USAGE_LOCK = threading.Lock()
_API_TL = threading.local()   # son başarılı API çağrısının süresi (ms): bekleme/yeniden deneme süreleri hariç
USAGE = defaultdict(lambda: Counter())
class BudgetExceeded(RuntimeError):
    pass
_BUDGET_LOGGED = [False]
def add_usage(model, usage):
    if usage is None:
        return
    with USAGE_LOCK:
        for k in ("input_tokens", "output_tokens", "cache_creation_input_tokens", "cache_read_input_tokens"):
            USAGE[model][k] += int(getattr(usage, k, 0) or 0)
        det = getattr(usage, "output_tokens_details", None)
        USAGE[model]["thinking_tokens"] += int(getattr(det, "thinking_tokens", 0) or 0) if det is not None else 0
        USAGE[model]["calls"] += 1
def usage_cost_usd(model):
    u = USAGE.get(model, {})
    p = PRICES_USD_PER_MTOK.get(model)
    if not p:
        return float("nan")
    return (u.get("input_tokens", 0) * p[0] + u.get("output_tokens", 0) * p[1] +
            u.get("cache_read_input_tokens", 0) * p[2] + u.get("cache_creation_input_tokens", 0) * p[3]) / 1e6
def total_cost_usd():
    return float(np.nansum([usage_cost_usd(m) for m in USAGE])) if USAGE else 0.0
def usage_summary(model):
    u = USAGE.get(model, {}); n = max(1, u.get("calls", 0))
    return (f"{u.get('calls', 0)} çağrı | çağrı başına ort. çıktı {u.get('output_tokens', 0) / n:.0f} token "
            f"(düşünme {u.get('thinking_tokens', 0) / n:.0f}) | giriş {u.get('input_tokens', 0) / n:.0f} + önbellek {u.get('cache_read_input_tokens', 0) / n:.0f} | "
            f"tahmini ${usage_cost_usd(model):.2f}")
def check_budget():
    if BUDGET_GUARD_USD is not None and total_cost_usd() >= BUDGET_GUARD_USD:
        if not _BUDGET_LOGGED[0]:
            _BUDGET_LOGGED[0] = True
            log(f"BÜTÇE FRENİ: tahmini maliyet ${total_cost_usd():.2f} >= {BUDGET_GUARD_USD} $. Yeni API çağrısı yapılmıyor; "
                f"devam etmek için BUDGET_GUARD_USD'yi artırıp hücreyi yeniden çalıştırın (tamamlanan parçalar Drive'da).")
        raise BudgetExceeded("bütçe aşıldı")

def thinking_param(model, mode):
    """Düşünme (thinking) denetimi: Sonnet 5.5 ön düşünmeyi 'between_tools' ile kapatır, Haiku 5.5 'disabled' kabul eder,
    Opus 5.5 / Fable 5.1'de düşünme kapatılamaz (yalnız effort ile sınırlanır)."""
    if mode != "minimal":
        return None
    if "sonnet-5-5" in model:
        return {"type": "between_tools"}
    if "haiku-5-5" in model:
        return {"type": "disabled"}
    return None

def extract_json(text):
    try:
        return json.loads(text)
    except Exception:
        s, e = text.find("{"), text.rfind("}")
        return json.loads(text[s:e + 1])

ANTHROPIC_CLIENT = None
SDK_SUPPORTS = {"temperature": False, "output_config": True}
if not SMOKE_TEST:
    import anthropic, inspect as _inspect
    ANTHROPIC_CLIENT = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY, max_retries=4, timeout=600)
    try:
        _params = set(_inspect.signature(ANTHROPIC_CLIENT.messages.create).parameters)
        SDK_SUPPORTS = {"temperature": "temperature" in _params, "output_config": "output_config" in _params}
    except Exception:
        pass
    log(f"anthropic SDK {anthropic.__version__}: temperature={'var' if SDK_SUPPORTS['temperature'] else 'yok (API varsayılanı kullanılır)'}, "
        f"output_config={'var' if SDK_SUPPORTS['output_config'] else 'yok (düz JSON istemi)'}")

def claude_json(system_text, user_text, schema, model, max_tokens=8000, temperature=None, retries=6, effort=None, thinking_mode=None):
    """Yapılandırılmış JSON çıktısı (output_config.format). Düşünme/effort denetimi, SDK imzasına göre parametre seçimi,
    bütçe freni, kesilen yanıtta max_tokens artırımı, 429/5xx için üstel bekleme. Desteklenmeyen parametre 400 verirse düşürülür."""
    last_err = None
    thinking_mode = THINKING_MODE if thinking_mode is None else thinking_mode
    use_temp = SDK_SUPPORTS["temperature"] and temperature is not None
    use_output_config = SDK_SUPPORTS["output_config"]
    think = thinking_param(model, thinking_mode)
    eff = effort
    for attempt in range(retries):
        check_budget()
        try:
            kwargs = dict(model=model, max_tokens=max_tokens,
                          system=[{"type": "text", "text": system_text, "cache_control": {"type": "ephemeral"}}],
                          messages=[{"role": "user", "content": user_text}])
            if use_temp:
                kwargs["temperature"] = temperature
            oc = {}
            if use_output_config:
                oc["format"] = {"type": "json_schema", "schema": schema}
            else:
                kwargs["messages"][0]["content"] += "\n\nReturn ONLY valid JSON matching this schema:\n" + json.dumps(schema)
            if eff and use_output_config:
                oc["effort"] = eff
            if oc:
                kwargs["output_config"] = oc
            if think:
                kwargs["thinking"] = think
            _t0 = time.perf_counter()
            resp = ANTHROPIC_CLIENT.messages.create(**kwargs)
            _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
            add_usage(model, getattr(resp, "usage", None))
            if resp.stop_reason == "max_tokens":
                max_tokens = int(max_tokens * 1.5)
                raise RuntimeError(f"Yanıt max_tokens sınırında kesildi, sınır {max_tokens} olarak artırıldı")
            if resp.stop_reason == "refusal":
                raise RuntimeError("Model yanıtı reddetti (refusal)")
            text = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
            return extract_json(text)
        except BudgetExceeded:
            raise
        except TypeError as e:   # SDK imzası beklenenden farklı: parametreyi düşür ve hemen yeniden dene
            last_err = e; msg = str(e)
            if "temperature" in msg and use_temp:
                log("SDK temperature parametresini kabul etmiyor, parametresiz devam ediliyor"); use_temp = False; SDK_SUPPORTS["temperature"] = False; continue
            if "output_config" in msg and use_output_config:
                log("SDK output_config parametresini kabul etmiyor, düz JSON istemine geçiliyor"); use_output_config = False; SDK_SUPPORTS["output_config"] = False; continue
            if "thinking" in msg and think:
                log("SDK thinking parametresini kabul etmiyor, parametresiz devam ediliyor"); think = None; continue
            raise
        except Exception as e:
            last_err = e; msg = str(e)
            if e.__class__.__name__ == "BadRequestError":
                if use_temp and "temperature" in msg:
                    log(f"{model}: API temperature kabul etmedi, parametresiz yeniden deneniyor"); use_temp = False; continue
                if think and "thinking" in msg:
                    log(f"{model}: thinking={think} kabul edilmedi ({msg[:120]}), parametresiz yeniden deneniyor"); think = None; continue
                if eff and "effort" in msg:
                    log(f"{model}: effort={eff} kabul edilmedi ({msg[:120]}), effort olmadan yeniden deneniyor"); eff = None; continue
                raise  # diğer 400 hatalarında yeniden denemenin anlamı yok
            wait = min(90, (2 ** attempt) + random.random() * 2)
            log(f"API hatası ({model}, deneme {attempt + 1}/{retries}): {msg[:160]} -> {wait:.0f} sn bekleniyor")
            time.sleep(wait)
    raise RuntimeError(f"API çağrısı başarısız: {last_err}")

# --- Sahte üretici (SMOKE_TEST) ------------------------------------------------------
_MOCK_OPEN_TR = ["Acil yardım!", "Lütfen duyun,", "Durum çok kötü,", "Yardım edin,", "Arkadaşlar", "SOS", "Bilgi:", "Komşular için yazıyorum,"]
_MOCK_OPEN_EN = ["Urgent help!", "Please hear us,", "Situation is bad,", "Help us,", "Friends", "SOS", "Info:", "Writing for my neighbours,"]
def mock_generate(specs):
    rng = random.Random(hash(specs[0]["id"]) % 10**6)
    out = []
    for s in specs:
        d = s["district"]
        tr_needs = ", ".join(s["needs_tr"]); en_needs = ", ".join(s["needs_en"])
        k = rng.randint(0, 999)
        tr = f"{rng.choice(_MOCK_OPEN_TR)} {d}'de {rng.randint(2, 40)} kişiyiz, {tr_needs} lazım. {s['persona']} {k}"
        en = f"{rng.choice(_MOCK_OPEN_EN)} we are {rng.randint(2, 40)} people in {d}, we need {en_needs}. {s['persona']} {k}"
        if s["hashtag"]:
            tr += " #deprem"; en += " #earthquake"
        out.append({"id": s["id"], "district": d, "message_tr": tr, "message_en": en})
    return {"records": out}

# --- Üretim çalıştırıcı (parça önbelleği + eş zamanlı çağrılar) ------------------------------
def generate_chunk(chunk_specs, part_path):
    if os.path.exists(part_path) and not FORCE_REGENERATE:
        with open(part_path, encoding="utf-8") as f:
            return json.load(f)
    t0 = time.perf_counter()
    if SMOKE_TEST:
        data = mock_generate(chunk_specs)
    else:
        try:
            data = claude_json(SYSTEM_PROMPT_GEN, build_user_prompt(chunk_specs), GEN_SCHEMA, GEN_MODEL,
                               max_tokens=max(3000, GEN_MAX_TOKENS_PER_RECORD * len(chunk_specs)), temperature=GEN_TEMPERATURE, effort=GEN_EFFORT)
        except BudgetExceeded:
            return None
    part = {"specs": chunk_specs, "records": data.get("records", []), "model": GEN_MODEL,
            "seconds": round(time.perf_counter() - t0, 1), "ts": datetime.datetime.now().isoformat()}
    with open(part_path, "w", encoding="utf-8") as f:
        json.dump(part, f, ensure_ascii=False)
    return part

def run_generation(specs, round_idx):
    chunks = [specs[i:i + GEN_RECORDS_PER_CALL] for i in range(0, len(specs), GEN_RECORDS_PER_CALL)]
    parts = [None] * len(chunks)
    with ThreadPoolExecutor(max_workers=GEN_WORKERS) as ex:
        futs = {ex.submit(generate_chunk, ch, os.path.join(PARTS_DIR, f"r{round_idx}_c{i:04d}.json")): i for i, ch in enumerate(chunks)}
        done = 0
        for fut in tqdm(as_completed(futs), total=len(futs), desc=f"Üretim turu {round_idx}"):
            i = futs[fut]
            try:
                parts[i] = fut.result()
            except Exception as e:
                log(f"Parça {i} başarısız: {e}")
            done += 1
            if not SMOKE_TEST and (done % 10 == 0 or done == len(futs)):
                log(f"Üretim {done}/{len(futs)} | {GEN_MODEL}: {usage_summary(GEN_MODEL)} | oturum toplamı ${total_cost_usd():.2f}")
    return [p for p in parts if p]

PHONE_RE = re.compile(r"\d{3}[\s.-]?\d{3}[\s.-]?\d{2}[\s.-]?\d{2}")
def validate_records(parts):
    """Model çıktısını plana karşı denetler. Döndürür: kabul edilen kayıtlar (spec + metinler), ret nedenleri."""
    accepted, reasons = [], Counter()
    for part in parts:
        spec_by_id = {s["id"]: s for s in part["specs"]}
        seen = set()
        for r in part["records"]:
            s = spec_by_id.get(r.get("id"))
            if s is None or r["id"] in seen:
                reasons["kimlik uyuşmazlığı"] += 1; continue
            seen.add(r["id"])
            tr, en = str(r.get("message_tr", "")).strip(), str(r.get("message_en", "")).strip()
            d = s["district"]
            if not (10 <= len(tr) <= 420 and 10 <= len(en) <= 420):
                reasons["uzunluk"] += 1; continue
            if tr_lower(d) not in tr_lower(tr) or tr_lower(d) not in tr_lower(en):
                reasons["ilçe adı yok"] += 1; continue
            others_tr = [x for x in detect_districts(tr) if x != d]
            others_en = [x for x in detect_districts(en) if x != d]
            if others_tr or others_en:
                reasons["başka ilçe adı"] += 1; continue
            if PHONE_RE.search(tr) or PHONE_RE.search(en) or "http" in en.lower() or "@" in en:
                reasons["telefon/URL/@"] += 1; continue
            accepted.append({**s, "message_tr": tr, "message_en": en, "gen_model": part["model"]})
    return accepted, reasons

# --- Tekrar eleme (anlamsal) ---------------------------------------------------------------
def _hash_embed(texts, dim=256):
    out = np.zeros((len(texts), dim), dtype=np.float32)
    for i, t in enumerate(texts):
        for tok in re.findall(r"\w+", tr_lower(t)):
            h = int(hashlib.md5(tok.encode()).hexdigest(), 16)
            out[i, h % dim] += 1.0 if (h >> 8) % 2 else -1.0
    return out
_DEDUP_MODEL = None
def dedup_embed(texts):
    global _DEDUP_MODEL
    if SMOKE_TEST:
        return _hash_embed(texts)
    if _DEDUP_MODEL is None:
        from sentence_transformers import SentenceTransformer
        _DEDUP_MODEL = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
    return _DEDUP_MODEL.encode(texts, batch_size=128, convert_to_numpy=True, show_progress_bar=False)

def deduplicate(records, threshold):
    if not records:
        return records, 0
    norm = lambda t: re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", tr_lower(t))).strip()
    keep, seen_norm = [], set()
    for r in records:
        key = norm(r["message_en"])
        if key in seen_norm:
            continue
        seen_norm.add(key); keep.append(r)
    E = dedup_embed([r["message_en"] for r in keep]).astype(np.float32)
    E /= (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)
    S = E @ E.T
    np.fill_diagonal(S, 0.0)
    drop = np.zeros(len(keep), dtype=bool)
    for j in range(len(keep)):
        if j and S[j, :j][~drop[:j]].size and S[j, :j][~drop[:j]].max() > threshold:
            drop[j] = True
    final = [r for r, dr in zip(keep, drop) if not dr]
    return final, len(records) - len(final)

# --- Ana üretim döngüsü: üret -> doğrula -> tekrar ele -> eksik ilçeleri tamamla -----------------
base_q, rem = divmod(N_MESSAGES, len(DISTRICTS))
_order = DISTRICTS[:]; random.Random(SEED).shuffle(_order)
TARGET_Q = {d: base_q + (1 if i < rem else 0) for i, d in enumerate(_order)}
accepted_all, round_idx, gen_t0 = [], 0, time.perf_counter()
quota = {d: math.ceil(TARGET_Q[d] * GEN_OVERSAMPLE) for d in DISTRICTS}
while round_idx < 4:
    specs = build_specs(quota, round_idx)
    log(f"Tur {round_idx}: {len(specs)} kayıt planlandı ({len(specs) // GEN_RECORDS_PER_CALL + 1} çağrı)")
    parts = run_generation(specs, round_idx)
    acc, reasons = validate_records(parts)
    log(f"Tur {round_idx}: {len(acc)} kabul, retler: {dict(reasons)}")
    accepted_all, n_dup = deduplicate(accepted_all + acc, DEDUP_COSINE_THRESHOLD)
    cnt = Counter(r["district"] for r in accepted_all)
    deficits = {d: TARGET_Q[d] - cnt[d] for d in DISTRICTS if cnt[d] < TARGET_Q[d]}
    log(f"Tur {round_idx}: tekrar elenen {n_dup}, toplam kabul {len(accepted_all)}, eksik ilçe sayısı {len(deficits)}")
    if not deficits:
        break
    quota = {d: math.ceil(v * 1.3) + 1 for d, v in deficits.items()}
    round_idx += 1

# İlçe kotasına göre kırp, kimlikleri sırala
by_d = defaultdict(list)
for r in accepted_all:
    by_d[r["district"]].append(r)
final_records = []
for d in DISTRICTS:
    final_records.extend(by_d[d][:TARGET_Q[d]])
    if len(by_d[d]) < TARGET_Q[d]:
        log(f"UYARI: {d} için {len(by_d[d])}/{TARGET_Q[d]} kayıt")
random.Random(SEED).shuffle(final_records)
log(f"Üretim bitti: {len(final_records)} kayıt, süre {(time.perf_counter() - gen_t0) / 60:.1f} dk")

# --- Bağımsız modelle etiket kalite kontrolü (QC) ----------------------------------------------
# Üretici modelden farklı bir model, her kaydı plana karşı denetler: atanan ihtiyaçların hepsi metinde
# (açık veya örtük) var mı, listeden başka bir ihtiyaç geçiyor mu, ilçe doğru mu. Sonuç raporlanır
# (kalite_kontrol.csv), QC_EXCLUDE_FAILED=True ise başarısızlar veri setinden çıkarılır.
SYSTEM_PROMPT_QC = f"""You are a strict annotator for a disaster-informatics dataset. Each item has a Turkish social media post, its English version, an assigned Istanbul district and assigned needs taken from a fixed needs list.
For each item decide:
- all_needs_expressed: true only if EVERY assigned need is conveyed by the post (explicitly, or by clear implication a human reader would make; e.g. a diabetic relative without injections implies insülin).
- missing: assigned needs that are NOT conveyed (empty list if none).
- extra: items from the needs list that the post clearly asks for but that were NOT assigned (empty list if none; ignore generic words like "yardım").
- district_ok: true if the assigned district is mentioned and no other district from the list is mentioned.
Needs list (Turkish (English)):
{NEEDS_REFERENCE}
Districts: {", ".join(DISTRICTS)}
Return only JSON {{"results": [{{"id", "all_needs_expressed", "missing", "extra", "district_ok"}}]}} with one entry per item, same ids."""
QC_SCHEMA = {"type": "object", "properties": {"results": {"type": "array", "items": {
    "type": "object", "properties": {"id": {"type": "string"}, "all_needs_expressed": {"type": "boolean"},
                                     "missing": {"type": "array", "items": {"type": "string"}}, "extra": {"type": "array", "items": {"type": "string"}},
                                     "district_ok": {"type": "boolean"}},
    "required": ["id", "all_needs_expressed", "missing", "extra", "district_ok"], "additionalProperties": False}}},
    "required": ["results"], "additionalProperties": False}

def qc_chunk(chunk, part_path):
    if os.path.exists(part_path) and not FORCE_REGENERATE:
        with open(part_path, encoding="utf-8") as f:
            return json.load(f)
    if SMOKE_TEST:
        rng = random.Random(part_path)
        data = {"results": [{"id": r["id"], "all_needs_expressed": rng.random() < 0.95, "missing": [], "extra": [], "district_ok": True} for r in chunk]}
    else:
        user = "Items (one JSON object per line):\n" + "\n".join(json.dumps({k: r[k] for k in ("id", "district", "needs_tr", "needs_en", "explicitness", "message_tr", "message_en")}, ensure_ascii=False) for r in chunk)
        data = claude_json(SYSTEM_PROMPT_QC, user, QC_SCHEMA, QC_MODEL, max_tokens=200 * len(chunk) + 1500, temperature=0.0, effort=QC_EFFORT)
    with open(part_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    return data

if RUN_LABEL_QC and final_records:
    qc_t0 = time.perf_counter()
    chunks = [final_records[i:i + GEN_RECORDS_PER_CALL] for i in range(0, len(final_records), GEN_RECORDS_PER_CALL)]
    qc_results = {}
    with ThreadPoolExecutor(max_workers=GEN_WORKERS) as ex:
        futs = {ex.submit(qc_chunk, ch, os.path.join(PARTS_DIR, f"qc_{QC_MODEL}_c{i:04d}.json")): i for i, ch in enumerate(chunks)}
        for fut in tqdm(as_completed(futs), total=len(futs), desc="Kalite kontrol"):
            try:
                for r in fut.result().get("results", []):
                    qc_results[r["id"]] = r
            except Exception as e:
                log(f"QC parçası başarısız: {e}")
    for r in final_records:
        q = qc_results.get(r["id"])
        r["qc_checked"] = q is not None
        r["qc_needs_ok"] = bool(q["all_needs_expressed"]) if q else None
        r["qc_missing"] = "; ".join(q.get("missing", [])) if q else ""
        r["qc_extra"] = "; ".join(q.get("extra", [])) if q else ""
        r["qc_district_ok"] = bool(q["district_ok"]) if q else None
        r["qc_ok"] = bool(q and q["all_needs_expressed"] and q["district_ok"] and not q.get("extra"))
    n_chk = sum(r["qc_checked"] for r in final_records)
    qc_summary = {"denetlenen": n_chk, "ihtiyaclar_tam": sum(bool(r["qc_needs_ok"]) for r in final_records),
                  "fazladan_ihtiyac_yok": sum(1 for r in final_records if r["qc_checked"] and not r["qc_extra"]),
                  "ilce_dogru": sum(bool(r["qc_district_ok"]) for r in final_records), "tumu_gecti": sum(r["qc_ok"] for r in final_records),
                  "qc_modeli": QC_MODEL, "sure_dk": round((time.perf_counter() - qc_t0) / 60, 1)}
    log(f"Kalite kontrol ({QC_MODEL}): {qc_summary}")
    pd.DataFrame([{k: r.get(k) for k in ("id", "district", "needs_tr", "explicitness", "qc_needs_ok", "qc_missing", "qc_extra", "qc_district_ok", "qc_ok", "message_tr", "message_en")}
                  for r in final_records]).to_csv(os.path.join(DIRS["results"], "kalite_kontrol.csv"), index=False, encoding="utf-8-sig")
    with open(os.path.join(DIRS["logs"], "kalite_kontrol_ozet.json"), "w", encoding="utf-8") as f:
        json.dump(qc_summary, f, ensure_ascii=False, indent=1)
    if QC_EXCLUDE_FAILED:
        before = len(final_records)
        final_records = [r for r in final_records if r["qc_ok"]]
        log(f"QC_EXCLUDE_FAILED: {before - len(final_records)} kayıt çıkarıldı, {len(final_records)} kaldı")
else:
    qc_summary = None
    for r in final_records:
        r.update({"qc_checked": False, "qc_needs_ok": None, "qc_missing": "", "qc_extra": "", "qc_district_ok": None, "qc_ok": None})
for m in USAGE:
    log(f"Token kullanımı {m}: {usage_summary(m)}")

# --- Veri çerçevesi (Message / English / label1-3 + ek sütunlar) ---------------------------------------
_dates = ["2025-01-01", "2025-01-02", "2025-01-03", "2025-01-04"]
rows = []
rng = random.Random(SEED + 7)
for i, r in enumerate(final_records):
    lat, lon = DISTRICT_COORDS[r["district"]]
    labels = r["needs_tr"] + [None] * (3 - len(r["needs_tr"]))
    hour = int(np.clip(rng.gauss(6, 5), 0, 23))
    rows.append({
        "id": i, "Message": r["message_tr"], "Time": rng.choices(_dates, weights=[0.4, 0.3, 0.2, 0.1])[0],
        "Location": f"{lat + rng.gauss(0, 0.01):.4f}, {lon + rng.gauss(0, 0.01):.4f}",
        "hour": f"{hour:02d}:{rng.randint(0, 59):02d}:{rng.randint(0, 59):02d}",
        "label1": labels[0], "label2": labels[1], "label3": labels[2],
        "English": r["message_en"], "district": r["district"], "n_needs": len(r["needs_tr"]),
        "explicitness": r["explicitness"], "persona": r["persona"], "register": r["register"], "length": r["length"],
        "typos": r["typos"], "no_diacritics": r["no_diacritics"], "hashtag": r["hashtag"], "emoji": r["emoji"],
        "neighbourhood": r["neighbourhood"], "numbers": r["numbers"], "gen_model": r["gen_model"], "gen_round": r["round"],
        "gen_id": r["id"], "source": "synthetic",
        "qc_needs_ok": r.get("qc_needs_ok"), "qc_missing": r.get("qc_missing", ""), "qc_extra": r.get("qc_extra", ""),
        "qc_district_ok": r.get("qc_district_ok"), "qc_ok": r.get("qc_ok"),
    })
df_syn = pd.DataFrame(rows)
for _c in ("label1", "label2", "label3"):
    df_syn[_c] = df_syn[_c].apply(lambda x: canon_need_tr(x) if isinstance(x, str) else x)
df_syn["labels_tr"] = df_syn[["label1", "label2", "label3"]].apply(lambda s: [x for x in s if isinstance(x, str)], axis=1)
df_syn["labels_en"] = df_syn["labels_tr"].apply(lambda L: list(dict.fromkeys(TR2EN[x] for x in L)))
df_syn["districts_detected"] = df_syn["Message"].apply(detect_districts)
DISTRICT_DETECT_AGREEMENT = float(df_syn.apply(lambda r: r["districts_detected"] == [r["district"]], axis=1).mean())
log(f"İlçe adı tespiti (özgün alt dize yaklaşımı) ile plan uyumu: {DISTRICT_DETECT_AGREEMENT:.3f}")

DATASET_BASENAME = f"sentetik_paylasimlar_{GEN_SIGNATURE}"
_export = df_syn.drop(columns=["labels_tr", "labels_en", "districts_detected"])
_export.to_csv(os.path.join(DATA_DIR, DATASET_BASENAME + ".csv"), index=False, encoding="utf-8-sig")
_export.to_excel(os.path.join(DATA_DIR, DATASET_BASENAME + ".xlsx"), index=False)
_export.to_excel(os.path.join(DIRS["results"], "veri_seti_sentetik.xlsx"), index=False)
with open(os.path.join(DATA_DIR, DATASET_BASENAME + ".jsonl"), "w", encoding="utf-8") as f:
    for r in final_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
with open(os.path.join(DIRS["logs"], "ilce_profilleri.json"), "w", encoding="utf-8") as f:
    json.dump(DISTRICT_PROFILE, f, ensure_ascii=False, indent=1)
log(f"Veri seti kaydedildi: {os.path.join(DATA_DIR, DATASET_BASENAME + '.xlsx')}")
print(df_syn[["district", "Message", "English", "label1", "label2", "label3", "explicitness"]].head(8).to_string())

In [ ]:
# =============================================================================
# 4. ÖN İŞLEME (düz / lemma / gövde), VERİ İSTATİSTİKLERİ
# =============================================================================
import inflect
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer, PorterStemmer
INFLECT = inflect.engine()
STOP_EN = set(stopwords.words("english"))
LEMM, STEMMER = WordNetLemmatizer(), PorterStemmer()
LETTERS_RE = re.compile(r"[^a-zA-ZğĞçÇşŞüÜöÖıİ]")   # Türkçe harfler korunur (ilçe adları parçalanmaz)

def numbers_to_words(text):
    return re.sub(r"\d+", lambda m: " " + (INFLECT.number_to_words(int(m.group())) if len(m.group()) < 12 else "") + " ", text)

def en_lower(s):
    """İngilizce metin için küçük harf: 'İ' -> 'i' (Python lower() birleşik nokta üretir), 'I' -> 'i' (Türkçe kural UYGULANMAZ)."""
    return str(s).replace("İ", "i").lower()

def plain_text(text):
    """Düz metin (İngilizce hat): sayılar sözcüğe, yalnız harfler, İngilizce küçük harf; Türkçe yer adlarındaki harfler korunur."""
    if not isinstance(text, str):
        return ""
    t = LETTERS_RE.sub(" ", numbers_to_words(text))
    return " ".join(en_lower(t).split())

def _wn_pos(tag):
    return {"J": wordnet.ADJ, "V": wordnet.VERB, "R": wordnet.ADV}.get(tag[:1], wordnet.NOUN)

_POS_OK = True
try:
    nltk.pos_tag(["test"])
except Exception as e:
    _POS_OK = False
    log("UYARI: POS etiketleyici yüklenemedi, lemmalama yalnız isim kipinde:", e)

def lemmatize_text(text):
    toks = [w for w in plain_text(text).split() if w not in STOP_EN]
    if _POS_OK:
        return " ".join(LEMM.lemmatize(w, _wn_pos(t)) for w, t in nltk.pos_tag(toks))
    return " ".join(LEMM.lemmatize(w) for w in toks)

def stem_text(text):
    return " ".join(STEMMER.stem(w) for w in plain_text(text).split() if w not in STOP_EN)

TEXT_FUNCS = {"plain": plain_text, "lemma": lemmatize_text, "stem": stem_text}
TEXT_TYPE_TR = {"plain": "Düz", "lemma": "Lemma", "stem": "Gövde"}

def add_text_columns(df):
    df = df.copy()
    for tt, fn in TEXT_FUNCS.items():
        df["text_" + tt] = df["English"].apply(fn)
    return df

def need_texts(text_type):
    """Model girdisi olarak ihtiyaç adları (PREPROCESS_NEEDS_LIKE_MESSAGES=True: mesajla aynı işlem)."""
    if PREPROCESS_NEEDS_LIKE_MESSAGES:
        out = [TEXT_FUNCS[text_type](n) for n in EN_LABELS]
        return [o if o.strip() else n.lower() for o, n in zip(out, EN_LABELS)]
    return [n.lower() for n in EN_LABELS]

df_syn = add_text_columns(df_syn)
EVAL_SETS = {"sentetik": df_syn}

df_syn.drop(columns=["labels_tr", "labels_en", "districts_detected"]).to_excel(os.path.join(DIRS["results"], "veri_seti_sentetik_islenmis.xlsx"), index=False)

# --- Tablo 2 benzeri örnekler ---------------------------------------------------------------
ex = df_syn.sample(min(8, len(df_syn)), random_state=SEED)[["district", "Message", "English", "text_plain", "text_lemma", "text_stem", "labels_tr"]]
ex = ex.rename(columns={"district": "İlçe", "Message": "Paylaşım (TR)", "English": "Paylaşım (EN)", "text_plain": "Düz", "text_lemma": "Lemma", "text_stem": "Gövde", "labels_tr": "Gerçek ihtiyaç(lar)"})
save_table(ex, "tablo2_onislenmis_ornekler")

# --- Veri istatistikleri ---------------------------------------------------------------------
stats_d = (df_syn.groupby("district").agg(mesaj=("id", "size"), ort_kelime_en=("English", lambda s: s.str.split().str.len().mean()))
           .reindex(DISTRICTS).fillna(0))
stats_d["en_sik_ihtiyac"] = [", ".join(f"{n} ({c})" for n, c in Counter(x for L in df_syn[df_syn.district == d]["labels_tr"] for x in L).most_common(3)) for d in DISTRICTS]
stats_d["profil"] = [", ".join(DISTRICT_PROFILE[d]) for d in DISTRICTS]
save_table(stats_d.reset_index().rename(columns={"district": "İlçe"}), "veri_ilce_dagilimi")

cnt_any = Counter(x for L in df_syn["labels_tr"] for x in L); cnt_first = Counter(df_syn["label1"])
stats_n = pd.DataFrame({"İhtiyaç (TR)": NEEDS_TR, "İhtiyaç (EN)": [TR2EN[n] for n in NEEDS_TR],
                        "etiket_sayisi": [cnt_any[n] for n in NEEDS_TR], "birinci_etiket": [cnt_first[n] for n in NEEDS_TR]})
save_table(stats_n.sort_values("etiket_sayisi", ascending=False), "veri_ihtiyac_dagilimi")

summary = {
    "kayit": len(df_syn), "ilce": df_syn["district"].nunique(), "ilce_basina_ort": round(len(df_syn) / 39, 1),
    "benzersiz_etiket_tr": df_syn["labels_tr"].explode().nunique(),
    "ort_karakter_tr": round(df_syn["Message"].str.len().mean(), 1), "ort_kelime_tr": round(df_syn["Message"].str.split().str.len().mean(), 1),
    "ort_karakter_en": round(df_syn["English"].str.len().mean(), 1), "ort_kelime_en": round(df_syn["English"].str.split().str.len().mean(), 1),
    "etiket_1_2_3_oran": " / ".join(fmt_tr(v, 2) for v in df_syn["n_needs"].value_counts(normalize=True).reindex([1, 2, 3]).fillna(0)),
    "acik_karma_ortuk_oran": " / ".join(fmt_tr(v, 2) for v in df_syn["explicitness"].value_counts(normalize=True).reindex(["explicit", "mixed", "implicit"]).fillna(0)),
    "ilce_tespit_uyumu": round(DISTRICT_DETECT_AGREEMENT, 3), "uretim_modeli": GEN_MODEL,
}
save_table(pd.DataFrame([summary]).T.reset_index().rename(columns={"index": "ölçü", 0: "değer"}), "veri_ozet", float_nd=1)
log("Veri özeti:", summary)

# Şekiller: ilçe dağılımı ve ihtiyaç dağılımı
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(range(39), stats_d["mesaj"].values, color=PALETTE[0])
ax.set_xticks(range(39)); ax.set_xticklabels(DISTRICTS, rotation=75, ha="right", fontsize=8)
ax.set_ylabel("Paylaşım sayısı"); ax.set_title("İlçelere göre simüle paylaşım sayısı")
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "sekil_veri_ilce_dagilimi.png"), dpi=300); plt.close(fig)

top = stats_n.sort_values("etiket_sayisi", ascending=True).tail(30)
fig, ax = plt.subplots(figsize=(7.5, 8))
ax.barh(top["İhtiyaç (TR)"], top["etiket_sayisi"], color=PALETTE[2])
ax.set_xlabel("Etiket sayısı (bir paylaşımda 1-3 ihtiyaç)"); ax.set_title("En sık 30 ihtiyaç etiketi")
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "sekil_veri_ihtiyac_dagilimi.png"), dpi=300); plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(df_syn["Message"].str.split().str.len(), bins=25, alpha=0.7, label="Türkçe", color=PALETTE[0])
ax.hist(df_syn["English"].str.split().str.len(), bins=25, alpha=0.7, label="İngilizce", color=PALETTE[1])
ax.set_xlabel("Kelime sayısı"); ax.set_ylabel("Paylaşım"); ax.legend(frameon=False); ax.set_title("Paylaşım uzunluğu dağılımı")
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "sekil_veri_uzunluk.png"), dpi=300); plt.close(fig)
log("Ön işleme ve veri istatistikleri tamam.")

In [ ]:
# =============================================================================
# 5. DİL TEMSİL MODELLERİ, KOSİNÜS BENZERLİĞİ, DOĞRULUK ÖLÇÜTLERİ, HARİTALAR
# -----------------------------------------------------------------------------
# Eşit koşullar:
#  * Tüm modeller aynı mesajları, aynı ihtiyaç listesini, aynı ön işlemeyi, aynı karar kuralını
#    (kosinüs benzerliğinde en yüksek ihtiyaç = top-1) ve aynı ölçütleri kullanır; hiçbir model eğitilmez/ayarlanmaz.
#  * Her modelin çıkarım ayarları (kaynak, eğitim verisi, boyut, parametre sayısı, cümle temsili,
#    ince ayar durumu, kasa duyarlılığı, OOV davranışı) Tablo 3b'ye yazılır.
#  * Ham/ince ayarlı eş çiftler (BERT-raw ↔ BERT-nli, ALBERT ↔ ALBERT-paraphrase) ince ayar etkisini ayırır.
#  * Temel çizgiler (TF-IDF, rastgele, çoğunluk) şans düzeyini gösterir.
#  * Gecikme tüm yerel modeller için aynı cihazda (CPU, tek mesaj) ölçülür; toplu GPU süresi ayrıca verilir.
# Uygulama ayrıntıları: ihtiyaç gömmeleri bir kez hesaplanır; sıfır vektörlü mesaj "tahminsiz" sayılır;
# makaledeki ilçe ölçütü aynen korunur, mesaj düzeyi hit@1/hit@3 ve bootstrap %95 GA eklenir.
# =============================================================================
MODEL_SPECS = {
    # makaledeki 7 model
    "BERT":        {"kind": "st",  "name": "bert-base-nli-mean-tokens"},
    "ALBERT":      {"kind": "st",  "name": "albert-base-v2"},
    "RoBERTa":     {"kind": "st",  "name": "roberta-base-nli-mean-tokens"},
    "GloVe":       {"kind": "glove"},
    "Word2Vec":    {"kind": "w2v"},
    "USE":         {"kind": "use"},
    "FastText":    {"kind": "fasttext"},
    # ek modeller (eşit koşul çiftleri ve güncel SBERT)
    "SBERT-mpnet":       {"kind": "st", "name": "all-mpnet-base-v2"},
    "BERT-raw":          {"kind": "st", "name": "bert-base-uncased"},
    "ALBERT-paraphrase": {"kind": "st", "name": "paraphrase-albert-small-v2"},
    # Türkçe metin + Türkçe ihtiyaç listesi (yalnız düz metin)
    "SBERT-çokdilli (TR)": {"kind": "st_tr", "name": "paraphrase-multilingual-MiniLM-L12-v2"},
    "BERTurk-STS (TR)":    {"kind": "st_tr", "name": "emrecan/bert-base-turkish-cased-mean-nli-stsb-tr"},
    # temel çizgiler
    "Temel: TF-IDF":   {"kind": "tfidf"},
    "Temel: Rastgele": {"kind": "random"},
    "Temel: Çoğunluk": {"kind": "majority"},
}
MODEL_META = {   # Tablo 3b için sabit bilgiler (boyut ve parametre sayısı çalışma anında ölçülür)
    "BERT":     dict(tur="Transformer (BERT-base)", egitim="Wikipedia + BookCorpus; SNLI+MultiNLI ile cümle benzerliği ince ayarı", temsil="Ortalama havuzlama (sentence-transformers)", ince_ayar="Evet (NLI)", kasa="Duyarsız", oov="WordPiece alt kelime"),
    "ALBERT":   dict(tur="Transformer (ALBERT-base-v2)", egitim="Wikipedia + BookCorpus; benzerlik ince ayarı YOK", temsil="Ortalama havuzlama (otomatik)", ince_ayar="Hayır", kasa="Duyarsız", oov="SentencePiece alt kelime"),
    "RoBERTa":  dict(tur="Transformer (RoBERTa-base)", egitim="160 GB web metni; SNLI+MultiNLI ince ayarı", temsil="Ortalama havuzlama (sentence-transformers)", ince_ayar="Evet (NLI)", kasa="Duyarlı", oov="Byte-BPE alt kelime"),
    "GloVe":    dict(tur="Statik kelime vektörü", egitim="Common Crawl, 840 milyar token", temsil="Sözlükteki kelime vektörlerinin ortalaması", ince_ayar="Hayır", kasa="Duyarlı", oov="Yok (kelime atlanır)"),
    "Word2Vec": dict(tur="Statik kelime vektörü", egitim="Google News, 100 milyar token", temsil="Sözlükteki kelime vektörlerinin ortalaması", ince_ayar="Hayır", kasa="Duyarlı", oov="Yok (kelime atlanır)"),
    "USE":      dict(tur="Cümle kodlayıcı (DAN, USE v4)", egitim="Wikipedia, web haber/forum, SNLI (çok görevli)", temsil="Doğrudan cümle gömmesi", ince_ayar="Hayır (cümle düzeyinde eğitilmiş)", kasa="Duyarsız", oov="Model içi sözlük/karma"),
    "FastText": dict(tur="Statik kelime vektörü (CBOW + karakter n-gram)", egitim="Common Crawl + Wikipedia (cc.en.300)", temsil="Birim normlu kelime vektörlerinin ortalaması", ince_ayar="Hayır", kasa="Duyarlı", oov="Karakter n-gram (her kelimeye vektör)"),
    "SBERT-mpnet":       dict(tur="Transformer (MPNet-base)", egitim="1 milyar+ cümle çifti ile karşıtsal ince ayar", temsil="Ortalama havuzlama", ince_ayar="Evet (cümle çiftleri)", kasa="Duyarsız", oov="WordPiece alt kelime"),
    "BERT-raw":          dict(tur="Transformer (BERT-base, ham)", egitim="Wikipedia + BookCorpus; ince ayar YOK (ALBERT ile eşit koşul)", temsil="Ortalama havuzlama (otomatik)", ince_ayar="Hayır", kasa="Duyarsız", oov="WordPiece alt kelime"),
    "ALBERT-paraphrase": dict(tur="Transformer (ALBERT-small-v2)", egitim="Paraphrase cümle çiftleri ile ince ayar (BERT-nli ile eşit koşul)", temsil="Ortalama havuzlama", ince_ayar="Evet (paraphrase)", kasa="Duyarsız", oov="SentencePiece alt kelime"),
    "SBERT-çokdilli (TR)": dict(tur="Transformer (MiniLM-L12, çok dilli damıtma)", egitim="50+ dilde paralel cümle çiftleri (Türkçe dâhil)", temsil="Ortalama havuzlama", ince_ayar="Evet (çok dilli damıtma)", kasa="Duyarlı", oov="SentencePiece alt kelime"),
    "BERTurk-STS (TR)":    dict(tur="Transformer (BERTurk-base)", egitim="Türkçe korpus; NLI-TR + STSb-TR ince ayarı", temsil="Ortalama havuzlama", ince_ayar="Evet (NLI/STS, Türkçe)", kasa="Duyarlı", oov="WordPiece alt kelime"),
    "Temel: TF-IDF":   dict(tur="Sözcüksel temel çizgi", egitim="Yalnız bu veri seti (1-2 gram TF-IDF)", temsil="Seyrek TF-IDF vektörü", ince_ayar="Hayır", kasa="Duyarsız", oov="Yok"),
    "Temel: Rastgele": dict(tur="Şans düzeyi", egitim="Yok", temsil="Rastgele ihtiyaç", ince_ayar="Hayır", kasa="-", oov="-"),
    "Temel: Çoğunluk": dict(tur="Şans düzeyi", egitim="Yok", temsil="Veri setindeki en sık ihtiyaç", ince_ayar="Hayır", kasa="-", oov="-"),
}
MAIN_MODELS = [m for m in MODELS_TO_RUN if m in MODEL_SPECS]
ALL_MODELS = [m for m in MODELS_TO_RUN + EXTRA_MODELS + TURKISH_MODELS + BASELINES if m in MODEL_SPECS]
BASELINE_NAMES = [m for m in BASELINES if m in MODEL_SPECS]

def find_existing(cands):
    for rel in cands:
        p = os.path.join(DRIVE_ROOT, rel)
        if os.path.exists(p):
            return p
    return None

def download_file(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 1e6:
        return dest
    log(f"İndiriliyor: {url}")
    tmp = dest + ".part"
    with tqdm(unit="B", unit_scale=True, desc=os.path.basename(dest)) as bar:
        def hook(b, bs, total):
            if total > 0:
                bar.total = total
            bar.update(bs)
        urllib.request.urlretrieve(url, tmp, reporthook=hook)
    os.replace(tmp, dest)
    return dest

# --- Gömme sınıfları ---------------------------------------------------------------------
class STEmbedder:
    def __init__(self, name):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(name, device=DEVICE)
        self.info = f"sentence-transformers:{name}"
        self.n_params = int(sum(p.numel() for p in self.model.parameters()))
        self.dim = int(self.model.get_sentence_embedding_dimension() or 0)
    def encode(self, texts):
        return np.asarray(self.model.encode(list(texts), batch_size=64, convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)
    def set_device(self, dev):
        try:
            self.model.to(dev)
        except Exception as e:
            log("cihaz değiştirilemedi:", e)
    def close(self):
        del self.model

class KVEmbedder:
    """Kelime vektörlerinin ortalaması (Word2Vec/GloVe: yalnız sözlükteki kelimeler; FastText: alt kelimelerle tüm kelimeler, birim normlu)."""
    def __init__(self, kv, info, fasttext=False):
        self.kv, self.info, self.fasttext = kv, info, fasttext
        self.dim = int(kv.vector_size)
        self.n_params = int(len(kv.key_to_index) * kv.vector_size)
        if fasttext and hasattr(kv, "vectors_ngrams"):
            self.n_params += int(np.prod(kv.vectors_ngrams.shape))
    def _vec(self, text):
        words = text.split()
        if self.fasttext:
            vs = []
            for w in words:
                v = self.kv.get_vector(w, norm=True)
                if np.isfinite(v).all() and np.linalg.norm(v) > 0:
                    vs.append(v)
        else:
            vs = [self.kv[w] for w in words if w in self.kv.key_to_index]
        return np.mean(vs, axis=0) if vs else np.zeros(self.dim, dtype=np.float32)
    def encode(self, texts):
        return np.vstack([self._vec(t) for t in texts]).astype(np.float32)
    def close(self):
        del self.kv

class USEEmbedder:
    def __init__(self, model, info):
        self.model, self.info = model, info
        try:
            self.n_params = int(sum(int(np.prod(v.shape)) for v in model.variables))
        except Exception:
            self.n_params = None
        self.dim = int(self.encode(["test"]).shape[1])
    def encode(self, texts):
        out = []
        for i in range(0, len(texts), 256):
            out.append(self.model(list(texts[i:i + 256])).numpy())
        return np.vstack(out).astype(np.float32)
    def close(self):
        del self.model

class TfidfEmbedder:
    """Sözcüksel temel çizgi: mesajlar + ihtiyaç adları üzerinde 1-2 gram TF-IDF, kosinüs."""
    def __init__(self):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.vec_cls = TfidfVectorizer; self.info = "TF-IDF (sklearn, 1-2 gram, sublinear tf)"; self.n_params = 0; self.dim = 0; self.vec = None
    def fit(self, texts):
        self.vec = self.vec_cls(ngram_range=(1, 2), sublinear_tf=True, min_df=1).fit(texts)
        self.dim = len(self.vec.vocabulary_)
    def encode(self, texts):
        return self.vec.transform(list(texts)).toarray().astype(np.float32)
    def close(self):
        pass

class MockEmbedder:
    """SMOKE_TEST: kelime karma (hash) vektörleri. Gerçek anlam taşımaz, yalnız akışı test eder."""
    def __init__(self, name):
        self.info = f"mock:{name}"; self.seed = int(hashlib.md5(name.encode()).hexdigest(), 16) % 1000
        self.n_params = 0; self.dim = 128 + self.seed % 7
    def encode(self, texts):
        return _hash_embed(texts, dim=self.dim)
    def set_device(self, dev):
        pass
    def close(self):
        pass

# --- Yükleyiciler ----------------------------------------------------------------------------
def glove_txt_to_kv(txt_path, dim=300):
    """glove.840B.300d.txt içindeki boşluklu/bozuk satırlara dayanıklı yükleyici (gensim'in load_word2vec_format'ı bu dosyada hata verir)."""
    from gensim.models import KeyedVectors
    n = 0
    with open(txt_path, "rb") as f:
        for _ in f:
            n += 1
    words, mat, seen, bad = [], np.empty((n, dim), dtype=np.float32), set(), 0
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in tqdm(f, total=n, desc="GloVe okunuyor"):
            parts = line.rstrip("\n").split(" ")
            if len(parts) <= dim:
                bad += 1; continue
            word = " ".join(parts[:-dim])
            if word in seen:
                bad += 1; continue
            try:
                vec = np.asarray(parts[-dim:], dtype=np.float32)
            except ValueError:
                bad += 1; continue
            seen.add(word); words.append(word); mat[len(words) - 1] = vec
    kv = KeyedVectors(vector_size=dim)
    kv.add_vectors(words, mat[:len(words)])
    log(f"GloVe yüklendi: {len(words)} kelime, atlanan satır {bad}")
    return kv

def load_model(name):
    spec = MODEL_SPECS[name]
    if spec["kind"] == "tfidf":
        return TfidfEmbedder()
    if SMOKE_TEST:
        return MockEmbedder(name)
    if spec["kind"] in ("st", "st_tr"):
        return STEmbedder(spec["name"])
    from gensim.models import KeyedVectors
    import gensim.downloader as gapi
    if spec["kind"] == "w2v":
        p = find_existing(MODEL_DIR_CANDIDATES["Word2Vec"])
        if p:
            return KVEmbedder(KeyedVectors.load_word2vec_format(p, binary=True), f"word2vec-google-news-300 ({p})")
        return KVEmbedder(gapi.load("word2vec-google-news-300"), "word2vec-google-news-300 (gensim-data)")
    if spec["kind"] == "glove":
        p = find_existing(MODEL_DIR_CANDIDATES["GloVe"])
        local_kv = os.path.join(LOCAL_SCRATCH, "glove.840B.300d.kv")
        if GLOVE_VARIANT == "840B":
            if p and p.endswith(".kv"):
                return KVEmbedder(KeyedVectors.load(p, mmap="r"), f"glove.840B.300d ({p})")
            if os.path.exists(local_kv):
                return KVEmbedder(KeyedVectors.load(local_kv, mmap="r"), "glove.840B.300d (yerel önbellek)")
            txt = p if (p and p.endswith(".txt")) else None
            if txt is None:
                zp = os.path.join(LOCAL_SCRATCH, "glove.840B.300d.zip")
                for url in ("https://nlp.stanford.edu/data/glove.840B.300d.zip",
                            "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.840B.300d.zip"):
                    try:
                        download_file(url, zp); break
                    except Exception as e:
                        log("GloVe indirme hatası:", url, e)
                if os.path.exists(zp):
                    with zipfile.ZipFile(zp) as z:
                        z.extractall(LOCAL_SCRATCH)
                    txt = os.path.join(LOCAL_SCRATCH, "glove.840B.300d.txt")
            if txt and os.path.exists(txt):
                kv = glove_txt_to_kv(txt, 300)
                kv.save(local_kv)
                if CACHE_MODELS_TO_DRIVE:
                    kv.save(os.path.join(MODEL_DIR, "glove.840B.300d.kv"))
                return KVEmbedder(kv, f"glove.840B.300d ({txt})")
            log("UYARI: GloVe 840B bulunamadı/indirilemedi, 'glove-wiki-gigaword-300' yedeği kullanılıyor")
        return KVEmbedder(gapi.load("glove-wiki-gigaword-300"), "glove-wiki-gigaword-300 (gensim-data, YEDEK)")
    if spec["kind"] == "fasttext":
        from gensim.models.fasttext import load_facebook_vectors
        p = find_existing(MODEL_DIR_CANDIDATES["FastText"])
        if not p:
            gz = download_file("https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz", os.path.join(LOCAL_SCRATCH, "cc.en.300.bin.gz"))
            p = os.path.join(LOCAL_SCRATCH, "cc.en.300.bin")
            if not os.path.exists(p):
                import gzip
                with gzip.open(gz, "rb") as fi, open(p, "wb") as fo:
                    shutil.copyfileobj(fi, fo, length=64 * 1024 * 1024)
            if CACHE_MODELS_TO_DRIVE:
                shutil.copy(p, os.path.join(MODEL_DIR, "cc.en.300.bin"))
        return KVEmbedder(load_facebook_vectors(p), f"cc.en.300.bin ({p})", fasttext=True)
    if spec["kind"] == "use":
        import tensorflow as tf
        try:
            tf.config.set_visible_devices([], "GPU")   # USE CPU'da çalışır; GPU belleği torch modellerine kalır
        except Exception:
            pass
        import tensorflow_hub as hub
        p = find_existing(MODEL_DIR_CANDIDATES["USE"])
        if p:
            return USEEmbedder(hub.load(p), f"universal-sentence-encoder/4 ({p})")
        last = None
        for url in ("https://tfhub.dev/google/universal-sentence-encoder/4",
                    "https://www.kaggle.com/models/google/universal-sentence-encoder/TensorFlow2/universal-sentence-encoder/2"):
            try:
                return USEEmbedder(hub.load(url), f"universal-sentence-encoder/4 ({url})")
            except Exception as e:
                last = e; log("USE yükleme hatası:", url, e)
        raise RuntimeError(f"USE yüklenemedi: {last}")
    raise ValueError(name)

# --- Ölçütler -----------------------------------------------------------------------------
def top_modes(counter):
    if not counter:
        return set()
    m = max(counter.values())
    return {k for k, v in counter.items() if v == m}

def district_metric_paper(df, preds):
    """Makaledeki ölçüt: ilçede en sık tahmin edilen ihtiyaç(lar) ile en sık gerçek ihtiyaç(lar) kesişiyorsa 1, değilse 0."""
    acc = {}
    for d, idx in df.groupby("district").indices.items():
        actual = Counter(x for i in idx for x in df["labels_en"].iat[i])
        predicted = Counter(preds[i] for i in idx if preds[i] is not None)
        if not actual or not predicted:
            continue
        acc[d] = 1 if (top_modes(actual) & top_modes(predicted)) else 0
    return acc

def district_top_pred(df, preds):
    out = {}
    for d, idx in df.groupby("district").indices.items():
        c = Counter(preds[i] for i in idx if preds[i] is not None)
        if c:
            out[d] = c.most_common(1)[0]
    return out

def bootstrap_ci(values, n_boot=BOOTSTRAP_N, seed=SEED):
    v = np.asarray(values, dtype=float)
    if len(v) == 0:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    means = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(axis=1)
    return (float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5)))

def evaluate_predictions(df, top1, top3, tag):
    labels = df["labels_en"].tolist()
    hit1 = np.array([(p is not None) and (p in L) for p, L in zip(top1, labels)], dtype=float)
    hit3 = np.array([any((p in L) for p in P if p is not None) for P, L in zip(top3, labels)], dtype=float)
    d_acc = district_metric_paper(df, top1)
    d_vals = list(d_acc.values())
    hit1_by_d = pd.Series(hit1, index=df["district"].values).groupby(level=0).mean()
    ci_d, ci_m = bootstrap_ci(d_vals), bootstrap_ci(hit1)
    res = {"ilce_dogruluk_makale": float(np.mean(d_vals)) if d_vals else float("nan"), "ilce_ga_alt": ci_d[0], "ilce_ga_ust": ci_d[1],
           "hit1": float(hit1.mean()), "hit1_ga_alt": ci_m[0], "hit1_ga_ust": ci_m[1],
           "hit1_ilce_ort": float(hit1_by_d.mean()), "hit3": float(hit3.mean()),
           "n_ilce": len(d_vals), "n_mesaj": len(df), "tahminsiz": int(sum(p is None for p in top1))}
    dtp = district_top_pred(df, top1)
    rows = []
    for d, idx in df.groupby("district").indices.items():
        actual = Counter(x for i in idx for x in df["labels_en"].iat[i])
        rows.append({"İlçe": d, "mesaj": len(idx), "gercek_en_sik": ", ".join(sorted(top_modes(actual))),
                     "tahmin_en_sik": dtp.get(d, ("", 0))[0], "tahmin_sayi": dtp.get(d, ("", 0))[1],
                     "dogru_makale_olcutu": d_acc.get(d, np.nan), "hit1_ort": float(hit1_by_d.get(d, np.nan))})
    pd.DataFrame(rows).to_csv(os.path.join(DIRS["preds"], f"ilce_{tag}.csv"), index=False, encoding="utf-8-sig")
    pd.DataFrame({"id": df["id"].values, "district": df["district"].values, "gercek": [";".join(L) for L in labels],
                  "top1": top1, "top2": [P[1] if len(P) > 1 else None for P in top3], "top3": [P[2] if len(P) > 2 else None for P in top3],
                  "hit1": hit1.astype(int)}).to_csv(os.path.join(DIRS["preds"], f"mesaj_{tag}.csv"), index=False, encoding="utf-8-sig")
    return res, hit1

def plain_text_tr(text):
    return " ".join(tr_lower(LETTERS_RE.sub(" ", str(text))).split())

def run_embedding_config(embedder, df, text_type, lang="en"):
    """Bir model × metin türü için top-1/top-3 tahminler ve süreler. lang='tr': Türkçe metin + Türkçe ihtiyaç adları."""
    if lang == "tr":
        msgs = df["Message"].apply(plain_text_tr).tolist()
        needs = [plain_text_tr(n) for n in NEEDS_TR]
        need_labels = [TR2EN[n] for n in NEEDS_TR]
    else:
        msgs = df["text_" + text_type].tolist()
        needs = need_texts(text_type)
        need_labels = list(EN_LABELS)
    if hasattr(embedder, "fit"):
        embedder.fit(msgs + needs)
    t0 = time.perf_counter(); M = embedder.encode(msgs); t_batch = (time.perf_counter() - t0) / max(1, len(msgs)) * 1000
    Nv = embedder.encode(needs)
    m_norm = np.linalg.norm(M, axis=1, keepdims=True); n_norm = np.linalg.norm(Nv, axis=1, keepdims=True)
    S = (M / (m_norm + 1e-9)) @ (Nv / (n_norm + 1e-9)).T
    zero = (m_norm[:, 0] == 0)
    order = np.argsort(-S, axis=1)
    top1, top3 = [], []
    for i in range(len(msgs)):
        if zero[i]:
            top1.append(None); top3.append([]); continue
        uniq = list(dict.fromkeys(need_labels[j] for j in order[i, :6]))[:3]   # aynı İngilizce etikete giden Türkçe adları birleştir
        top1.append(uniq[0]); top3.append(uniq)
    # tek mesaj gecikmesi: tüm yerel modeller için aynı cihazda (varsayılan CPU)
    if LATENCY_ON_CPU and hasattr(embedder, "set_device"):
        embedder.set_device("cpu")
    sample = random.Random(SEED).sample(msgs, min(30, len(msgs)))
    lat = []
    for s in sample:
        t0 = time.perf_counter(); embedder.encode([s]); lat.append((time.perf_counter() - t0) * 1000)
    if LATENCY_ON_CPU and hasattr(embedder, "set_device"):
        embedder.set_device(DEVICE)
    return top1, top3, t_batch, float(np.median(lat))

def baseline_predictions(kind, df, seed=SEED):
    rng = random.Random(seed)
    if kind == "random":
        top3 = [rng.sample(EN_LABELS, 3) for _ in range(len(df))]
        return [t[0] for t in top3], top3
    freq = Counter(L[0] for L in df["labels_en"] if L)
    top = [n for n, _ in freq.most_common(3)]
    return [top[0]] * len(df), [top] * len(df)

# --- Haritalar -------------------------------------------------------------------------------
import folium
def need_color_map(needs):
    cmap = plt.get_cmap("tab20").colors + plt.get_cmap("tab20b").colors + plt.get_cmap("tab20c").colors
    return {n: matplotlib.colors.to_hex(cmap[i % len(cmap)]) for i, n in enumerate(sorted(set(needs)))}

def save_folium_map(dtp, path, title):
    m = folium.Map(location=[41.05, 28.95], zoom_start=10, tiles="cartodbpositron")
    colors = need_color_map([v[0] for v in dtp.values()])
    for d, (need, cnt) in dtp.items():
        lat, lon = DISTRICT_COORDS[d]
        folium.CircleMarker(location=[lat, lon], radius=9, color=colors[need], fill=True, fill_color=colors[need], fill_opacity=0.85,
                            popup=f"{d}: {EN2TR.get(need, need)} ({cnt})", tooltip=f"{d}: {EN2TR.get(need, need)}").add_to(m)
    legend = "<br>".join(f'<i style="background:{c};width:12px;height:12px;border-radius:50%;display:inline-block"></i> {EN2TR.get(n, n)}' for n, c in colors.items())
    m.get_root().html.add_child(folium.Element(
        f'<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:8px;border:1px solid grey;font-size:12px;max-height:400px;overflow:auto"><b>{title}</b><br>{legend}</div>'))
    m.save(path)

# İlçe poligonları (varsa) bir kez yüklenir
from matplotlib.patches import Polygon as MplPolygon
DISTRICT_POLYS = {}
if GEOJSON_PATH and os.path.exists(GEOJSON_PATH):
    with open(GEOJSON_PATH, encoding="utf-8") as fh:
        for f in json.load(fh)["features"]:
            g = f["geometry"]; rings = g["coordinates"] if g["type"] == "Polygon" else [r for poly in g["coordinates"] for r in poly]
            DISTRICT_POLYS[f["properties"]["ilce"]] = [np.array(r) for r in rings if len(r) >= 3]

def draw_static_map(ax, dtp, title):
    colors = need_color_map([v[0] for v in dtp.values()])
    for d in DISTRICTS:
        lat, lon = DISTRICT_COORDS[d]
        fc = colors[dtp[d][0]] if d in dtp else "white"
        if d in DISTRICT_POLYS:
            for ring in DISTRICT_POLYS[d]:
                ax.add_patch(MplPolygon(ring[:, :2], closed=True, facecolor=fc, edgecolor="#444444", linewidth=0.35, alpha=0.9, zorder=2))
            ax.annotate(d, (lon, lat), fontsize=3.6, ha="center", va="center", zorder=4)
        else:
            ax.scatter(lon, lat, s=60 if d in dtp else 22, color=fc, edgecolor="black" if d in dtp else "grey", linewidth=0.4, zorder=3)
            ax.annotate(d, (lon, lat), fontsize=3.8, xytext=(2, 2), textcoords="offset points", zorder=4)
    if DISTRICT_POLYS:
        ax.set_xlim(27.95, 29.95); ax.set_ylim(40.78, 41.60)
    else:
        ax.set_xlim(28.15, 29.75); ax.set_ylim(40.74, 41.30)
    ax.set_title(title, fontsize=8.5); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_linewidth(0.5)
    ax.set_aspect(1 / math.cos(math.radians(41)))
    handles = [plt.Line2D([], [], marker="o", color=c, linestyle="", markersize=4.5, label=EN2TR.get(n, n)) for n, c in colors.items()]
    ax.legend(handles=handles, fontsize=4.6, loc="upper center", bbox_to_anchor=(0.5, -0.02), frameon=False, ncol=4,
              handletextpad=0.3, columnspacing=0.8, borderaxespad=0.0)

# --- Ana değerlendirme döngüsü ----------------------------------------------------------------
RESULTS, HIT1_VECTORS, DISTRICT_PREDS, MODEL_INFO, MODEL_RUNTIME = [], {}, {}, {}, {}
for mname in ALL_MODELS:
    kind = MODEL_SPECS[mname]["kind"]
    log(f"=== Model: {mname} ===")
    if kind in ("random", "majority"):
        for set_name, dfe in EVAL_SETS.items():
            tag = f"{set_name}_{mname}_plain"
            top1, top3 = baseline_predictions(kind, dfe)
            res, hit1 = evaluate_predictions(dfe, top1, top3, tag)
            res.update({"veri": set_name, "model": mname, "metin": "plain", "ms_mesaj_toplu": 0.0, "ms_mesaj_tekil": 0.0})
            RESULTS.append(res); HIT1_VECTORS[tag] = hit1; DISTRICT_PREDS[tag] = district_top_pred(dfe, top1)
            log(f"{tag}: ilçe doğruluğu (makale) {res['ilce_dogruluk_makale']:.3f} | hit@1 {res['hit1']:.3f}")
        MODEL_INFO[mname] = "temel çizgi"; MODEL_RUNTIME[mname] = {"dim": 0, "n_params": 0}
        continue
    try:
        t0 = time.perf_counter(); emb = load_model(mname); MODEL_INFO[mname] = emb.info
        MODEL_RUNTIME[mname] = {"dim": getattr(emb, "dim", None), "n_params": getattr(emb, "n_params", None)}
        log(f"{mname} yüklendi ({emb.info}) {time.perf_counter() - t0:.0f} sn")
    except Exception as e:
        log(f"HATA: {mname} yüklenemedi, atlanıyor: {e}")
        MODEL_INFO[mname] = f"YÜKLENEMEDİ: {e}"
        continue
    text_types = ["plain"] if kind == "st_tr" else TEXT_TYPES
    for set_name, dfe in EVAL_SETS.items():
        for tt in text_types:
            tag = f"{set_name}_{mname}_{tt}"
            try:
                top1, top3, t_batch, t_single = run_embedding_config(emb, dfe, tt, lang="tr" if kind == "st_tr" else "en")
                res, hit1 = evaluate_predictions(dfe, top1, top3, tag)
                res.update({"veri": set_name, "model": mname, "metin": tt, "ms_mesaj_toplu": t_batch, "ms_mesaj_tekil": t_single})
                RESULTS.append(res); HIT1_VECTORS[tag] = hit1; DISTRICT_PREDS[tag] = district_top_pred(dfe, top1)
                save_folium_map(DISTRICT_PREDS[tag], os.path.join(DIRS["maps"], f"harita_{tag}.html"), f"{mname} / {TEXT_TYPE_TR[tt]} ({set_name})")
                log(f"{tag}: ilçe doğruluğu (makale) {res['ilce_dogruluk_makale']:.3f} | hit@1 {res['hit1']:.3f} | hit@3 {res['hit3']:.3f} | tahminsiz {res['tahminsiz']} | {t_single:.1f} ms/mesaj")
            except Exception as e:
                log(f"HATA: {tag} değerlendirilemedi: {e}")
        if MODEL_RUNTIME[mname].get("dim") in (None, 0):
            MODEL_RUNTIME[mname]["dim"] = getattr(emb, "dim", None)
    emb.close(); del emb; gc.collect()
    if torch is not None and DEVICE == "cuda":
        torch.cuda.empty_cache()

RES = pd.DataFrame(RESULTS)
RES.to_csv(os.path.join(DIRS["results"], "sonuclar_tum_konfigurasyonlar.csv"), index=False, encoding="utf-8-sig")
with open(os.path.join(DIRS["logs"], "model_bilgisi.json"), "w", encoding="utf-8") as f:
    json.dump({"kaynak": MODEL_INFO, "olcum": MODEL_RUNTIME}, f, ensure_ascii=False, indent=1)

# --- Tablo 3b: model ayarları (eşit koşullar belgesi) ---------------------------------------------
rows = []
for m in ALL_MODELS:
    meta = MODEL_META.get(m, {}); rt = MODEL_RUNTIME.get(m, {})
    npar = rt.get("n_params")
    rows.append({"Model": m, "Tür": meta.get("tur", ""), "Eğitim verisi / ince ayar": meta.get("egitim", ""),
                 "Cümle temsili": meta.get("temsil", ""), "Benzerlik ince ayarı": meta.get("ince_ayar", ""),
                 "Boyut": rt.get("dim", ""), "Parametre (M)": "" if not npar else round(npar / 1e6, 1),
                 "Kasa": meta.get("kasa", ""), "OOV": meta.get("oov", ""),
                 "Metin türleri": "Düz (Türkçe)" if MODEL_SPECS[m]["kind"] == "st_tr" else ("Düz" if MODEL_SPECS[m]["kind"] in ("random", "majority") else "Düz, Lemma, Gövde"),
                 "Karar kuralı": "kosinüs top-1" if MODEL_SPECS[m]["kind"] not in ("random", "majority") else meta.get("temsil", ""),
                 "Kaynak": MODEL_INFO.get(m, "")})
save_table(pd.DataFrame(rows), "tablo3b_model_ayarlari")

def pivot_table(set_name, metric, with_ci=None, models=None):
    sub = RES[RES.veri == set_name]
    models = [m for m in (models or ALL_MODELS) if m in sub.model.values]
    pv = sub.pivot(index="model", columns="metin", values=metric).reindex(index=models, columns=TEXT_TYPES)
    pv.columns = [TEXT_TYPE_TR[c] for c in pv.columns]
    if with_ci:
        lo = sub.pivot(index="model", columns="metin", values=with_ci[0]).reindex(index=models, columns=TEXT_TYPES)
        hi = sub.pivot(index="model", columns="metin", values=with_ci[1]).reindex(index=models, columns=TEXT_TYPES)
        pretty = pv.copy().astype(object)
        for i in range(len(models)):
            for j in range(len(TEXT_TYPES)):
                v, l, h = pv.iloc[i, j], lo.iloc[i, j], hi.iloc[i, j]
                pretty.iloc[i, j] = "" if pd.isna(v) else f"{fmt_tr(v)} [{fmt_tr(l)}; {fmt_tr(h)}]"
        return pv, pretty
    return pv, None

TABLES = {}
for set_name in EVAL_SETS:
    pv, pretty = pivot_table(set_name, "ilce_dogruluk_makale", ("ilce_ga_alt", "ilce_ga_ust"))
    TABLES[f"tablo4_ilce_dogrulugu_{set_name}"] = pv; save_table(pv.reset_index().rename(columns={"model": "Model"}), f"tablo4_ilce_dogrulugu_{set_name}")
    save_table(pretty.reset_index().rename(columns={"model": "Model"}), f"tablo4_ilce_dogrulugu_{set_name}_GA")
    pv, pretty = pivot_table(set_name, "hit1", ("hit1_ga_alt", "hit1_ga_ust"))
    TABLES[f"tablo4b_hit1_{set_name}"] = pv; save_table(pv.reset_index().rename(columns={"model": "Model"}), f"tablo4b_mesaj_hit1_{set_name}")
    save_table(pretty.reset_index().rename(columns={"model": "Model"}), f"tablo4b_mesaj_hit1_{set_name}_GA")
    pv, _ = pivot_table(set_name, "hit3"); TABLES[f"tablo4c_hit3_{set_name}"] = pv
    save_table(pv.reset_index().rename(columns={"model": "Model"}), f"tablo4c_mesaj_hit3_{set_name}")
    pv, _ = pivot_table(set_name, "hit1_ilce_ort"); save_table(pv.reset_index().rename(columns={"model": "Model"}), f"tablo4d_hit1_ilce_ortalamasi_{set_name}")
    pv, _ = pivot_table(set_name, "tahminsiz"); save_table(pv.reset_index().rename(columns={"model": "Model"}), f"tablo4e_tahminsiz_mesaj_{set_name}")
    print(f"\n[{set_name}] Tablo 4, ilçe doğruluğu (makale ölçütü):\n", TABLES[f"tablo4_ilce_dogrulugu_{set_name}"].round(3).to_string())
    print(f"\n[{set_name}] Tablo 4b, mesaj düzeyi hit@1:\n", TABLES[f"tablo4b_hit1_{set_name}"].round(3).to_string())

timing = (RES[(RES.veri == "sentetik") & (~RES.model.isin(BASELINE_NAMES))].groupby("model")[["ms_mesaj_toplu", "ms_mesaj_tekil"]].median()
          .reindex([m for m in ALL_MODELS if m in RES.model.values and m not in BASELINE_NAMES]).reset_index().rename(columns={"model": "Model"}))
timing["toplu_cihaz"] = ["CPU" if MODEL_SPECS[m]["kind"] in ("w2v", "glove", "fasttext", "use", "tfidf") else GPU_NAME for m in timing["Model"]]
timing["tekil_cihaz"] = ["CPU" if (LATENCY_ON_CPU or MODEL_SPECS[m]["kind"] in ("w2v", "glove", "fasttext", "use", "tfidf")) else GPU_NAME for m in timing["Model"]]
timing["kaynak"] = [MODEL_INFO.get(m, "") for m in timing["Model"]]
save_table(timing, "tablo6_cikarim_suresi")

# Statik çoklu harita: makaledeki modeller için (hit@1'e göre) en iyi metin türü + gerçek ihtiyaç haritası
for set_name, dfe in EVAL_SETS.items():
    sub = RES[(RES.veri == set_name) & (RES.model.isin(MAIN_MODELS))]
    if sub.empty:
        continue
    best = sub.sort_values("hit1", ascending=False).groupby("model").head(1)
    panels = [("Gerçek (en sık etiket)", {d: (sorted(top_modes(Counter(x for i in idx for x in dfe['labels_en'].iat[i])))[0], len(idx))
                                          for d, idx in dfe.groupby("district").indices.items()})]
    for _, r in best.iterrows():
        tag = f"{set_name}_{r['model']}_{r['metin']}"
        if tag in DISTRICT_PREDS:
            panels.append((f"{r['model']} / {TEXT_TYPE_TR[r['metin']]} (hit@1 {fmt_tr(r['hit1'])})", DISTRICT_PREDS[tag]))
    ncol = 3; nrow = math.ceil(len(panels) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.4 * ncol, 4.3 * nrow))
    axes = np.array(axes).reshape(-1)
    for ax, (title, dtp) in zip(axes, panels):
        draw_static_map(ax, dtp, title)
    for ax in axes[len(panels):]:
        ax.axis("off")
    fig.suptitle(f"İlçelere göre en sık tespit edilen ihtiyaç ({set_name} veri seti)", fontsize=12, y=0.995)
    fig.tight_layout(rect=(0, 0, 1, 0.98)); fig.savefig(os.path.join(DIRS["maps"], f"harita_coklu_{set_name}.png"), dpi=300); plt.close(fig)
    if len(panels) >= 2:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
        for ax, (title, dtp) in zip(axes, panels[:2]):
            draw_static_map(ax, dtp, title)
        fig.tight_layout(); fig.savefig(os.path.join(DIRS["maps"], f"harita_en_iyi_{set_name}.png"), dpi=300); plt.close(fig)
log("Model değerlendirmesi tamam.")

In [ ]:
# =============================================================================
# 6. BDM DEĞERLENDİRMESİ (Tablo 5): sıfır atış istem, makaledeki protokolün API karşılığı
# =============================================================================
import difflib
LLM_PROMPT_VERSION = "v1"
SYSTEM_PROMPT_LLM = ("You are a disaster response assistant. Based on the social media message of an earthquake victim, "
                     "identify the single most urgently needed relief item from the following list of disaster relief needs:\n"
                     + "\n".join(f"- {n}" for n in EN_LABELS) +
                     "\nFor each message respond only with the name of the need, exactly as written in the list.")
LLM_SCHEMA = {"type": "object", "properties": {"predictions": {"type": "array", "items": {
    "type": "object", "properties": {"id": {"type": "integer"}, "need": {"type": "string", "enum": EN_LABELS}},
    "required": ["id", "need"], "additionalProperties": False}}}, "required": ["predictions"], "additionalProperties": False}
with open(os.path.join(DIRS["logs"], "prompt_bdm.txt"), "w", encoding="utf-8") as f:
    f.write(SYSTEM_PROMPT_LLM)

def strip_additional(schema):
    if isinstance(schema, dict):
        return {k: strip_additional(v) for k, v in schema.items() if k != "additionalProperties"}
    if isinstance(schema, list):
        return [strip_additional(x) for x in schema]
    return schema

_LABEL_LOWER = {l.lower(): l for l in EN_LABELS}
def normalize_need(x):
    if not isinstance(x, str):
        return None
    k = x.strip().lower()
    if k in _LABEL_LOWER:
        return _LABEL_LOWER[k]
    m = difflib.get_close_matches(k, list(_LABEL_LOWER), n=1, cutoff=0.6)
    return _LABEL_LOWER[m[0]] if m else None

def build_llm_user(batch):
    if len(batch) == 1:
        return f"Message (id {batch[0][0]}): {batch[0][1]}\nReturn JSON {{\"predictions\":[{{\"id\":{batch[0][0]},\"need\":...}}]}}."
    return "Messages:\n" + "\n".join(f"[id {i}] {t}" for i, t in batch) + "\nReturn JSON {\"predictions\":[{\"id\":..,\"need\":..}, ...]} with one entry per message id."

# --- Hız sınırı (429) ve kota yönetimi -------------------------------------------------------
class LLMQuotaExhausted(Exception):
    """Günlük istek sınırı ya da kredi/kota bitti: bu model için yeni istek göndermenin anlamı yok."""

class _Pacer:
    """Sağlayıcı başına istek aralığı (RPM) ve 429 sonrası ortak bekleme."""
    def __init__(self, rpm=None):
        self.lock = threading.Lock(); self.next_ok = 0.0; self.min_interval = (60.0 / rpm) if rpm else 0.0
    def wait(self):
        with self.lock:
            now = time.monotonic(); delay = max(0.0, self.next_ok - now)
            self.next_ok = max(now, self.next_ok) + self.min_interval
        if delay > 0:
            time.sleep(delay)
    def hold(self, seconds):
        with self.lock:
            self.next_ok = max(self.next_ok, time.monotonic() + seconds)

_PACERS = {}
def pacer_for(provider):
    if provider not in _PACERS:
        _PACERS[provider] = _Pacer(LLM_EVAL_RPM_BY_PROVIDER.get(provider))
    return _PACERS[provider]

def _retry_after_seconds(msg):
    m = re.search(r"try again in\s*([\d.]+)\s*(ms|s|m)\b", msg, flags=re.I)
    if not m:
        return None
    v = float(m.group(1)); u = m.group(2).lower()
    return v / 1000 if u == "ms" else (v * 60 if u == "m" else v)

def classify_api_error(e):
    """'quota' (kredi/kota bitti), 'daily' (günlük istek sınırı), 'rate' (dakikalık sınır), 'other'."""
    msg = str(e); low = msg.lower()
    is429 = ("429" in msg) or ("rate limit" in low) or ("resource_exhausted" in low) or ("quota" in low) or type(e).__name__ in ("RateLimitError", "ResourceExhausted")
    if any(k in low for k in ("insufficient_quota", "no credits remaining", "credit_balance_exhausted", "spend_limit_exceeded", "usage_limit_exceeded", "billing")):
        return "quota"
    if is429 and any(k in low for k in ("per day", "requests per day", "tokens per day", "rpd", "perday", "daily")):
        return "daily"
    if is429:
        return "rate"
    return "other"

def llm_call_paced(provider, model, batch, max_attempts=8):
    """llm_call'ı hız sınırına uyarak çağırır: dakikalık 429'da bekleyip yeniden dener; kota/günlük sınırda LLMQuotaExhausted fırlatır."""
    pacer = pacer_for(provider)
    for attempt in range(max_attempts):
        if not SMOKE_TEST:
            pacer.wait()
        try:
            _API_TL.last_ms = float("nan")
            preds = llm_call(provider, model, batch)
            return preds, float(getattr(_API_TL, "last_ms", float("nan")))
        except LLMQuotaExhausted:
            raise
        except BudgetExceeded:
            raise
        except Exception as e:
            kind = classify_api_error(e)
            if kind in ("quota", "daily"):
                raise LLMQuotaExhausted(f"{kind}: {str(e)[:300]}")
            wait_s = _retry_after_seconds(str(e))
            if kind == "rate":
                wait_s = wait_s if wait_s is not None else min(60.0, 3.0 * (2 ** attempt))
                pacer.hold(wait_s + 0.5)
                time.sleep(wait_s + 0.5 + random.random())
                continue
            if attempt >= 2:
                raise
            time.sleep(2.0 * (attempt + 1))
    raise RuntimeError(f"{model}: {max_attempts} denemede hız sınırı aşılamadı")

_PROVIDER_OPTS = {"openai_reasoning": "minimal", "google_thinking_budget": 0}
OTHER_USAGE = defaultdict(lambda: Counter())
def _note_usage(model, u):
    """OpenAI / Google token sayaçları. Faturalanan çıktı: OpenAI'de completion_tokens (düşünme dâhil), Gemini'de candidates + thoughts."""
    if u is None:
        return
    with USAGE_LOCK:
        c = OTHER_USAGE[model]; c["calls"] += 1
        if hasattr(u, "prompt_tokens"):            # OpenAI
            det = getattr(u, "completion_tokens_details", None); pdt = getattr(u, "prompt_tokens_details", None)
            out = int(getattr(u, "completion_tokens", 0) or 0); think = int(getattr(det, "reasoning_tokens", 0) or 0) if det is not None else 0
            c["input_tokens"] += int(getattr(u, "prompt_tokens", 0) or 0); c["cached_input_tokens"] += int(getattr(pdt, "cached_tokens", 0) or 0) if pdt is not None else 0
            c["output_tokens"] += out - think; c["thinking_tokens"] += think; c["billed_output_tokens"] += out
        else:                                       # Google
            out = int(getattr(u, "candidates_token_count", 0) or 0); think = int(getattr(u, "thoughts_token_count", 0) or 0)
            c["input_tokens"] += int(getattr(u, "prompt_token_count", 0) or 0); c["cached_input_tokens"] += int(getattr(u, "cached_content_token_count", 0) or 0)
            c["output_tokens"] += out; c["thinking_tokens"] += think; c["billed_output_tokens"] += out + think

def cost_from_counter(model, u):
    """Sayaç sözlüğünden tahmini maliyet (USD, liste fiyatı)."""
    p = PRICES_USD_PER_MTOK.get(model)
    if not p or not u:
        return float("nan")
    if model.startswith("claude"):
        return (u.get("input_tokens", 0) * p[0] + u.get("output_tokens", 0) * p[1] + u.get("cache_read_input_tokens", 0) * p[2] + u.get("cache_creation_input_tokens", 0) * p[3]) / 1e6
    cached = u.get("cached_input_tokens", 0)
    return ((u.get("input_tokens", 0) - cached) * p[0] + cached * p[2] + u.get("billed_output_tokens", u.get("output_tokens", 0)) * p[1]) / 1e6

def other_usage_summary(model):
    u = OTHER_USAGE.get(model, {}); n = max(1, u.get("calls", 0)); cost = cost_from_counter(model, u)
    return (f"{u.get('calls', 0)} çağrı | çağrı başına giriş {u.get('input_tokens', 0) / n:.0f} (önbellek {u.get('cached_input_tokens', 0) / n:.0f}), çıktı {u.get('output_tokens', 0) / n:.0f} token "
            f"(düşünme {u.get('thinking_tokens', 0) / n:.0f})" + (f" | tahmini ${cost:.2f}" if cost == cost else ""))
def other_usage_cost(model):
    return cost_from_counter(model, OTHER_USAGE.get(model, {}))

_OPENAI_CLIENT = _GOOGLE_CLIENT = None
def llm_call(provider, model, batch):
    """batch: [(id, text), ...] -> {id: need}"""
    global _OPENAI_CLIENT, _GOOGLE_CLIENT
    user = build_llm_user(batch)
    if SMOKE_TEST:
        _API_TL.last_ms = 0.0
        rng = random.Random(batch[0][0] + hash(model) % 1000)
        return {i: rng.choice(EN_LABELS) for i, _ in batch}
    if provider == "anthropic":
        data = claude_json(SYSTEM_PROMPT_LLM, user, LLM_SCHEMA, model, max_tokens=150 * len(batch) + 1500, temperature=0.0, effort=LLM_EVAL_EFFORT)
    elif provider == "openai":
        if _OPENAI_CLIENT is None:
            from openai import OpenAI
            _OPENAI_CLIENT = OpenAI(api_key=OPENAI_API_KEY, max_retries=0)
        kwargs = dict(model=model, messages=[{"role": "system", "content": SYSTEM_PROMPT_LLM}, {"role": "user", "content": user}],
                      response_format={"type": "json_schema", "json_schema": {"name": "preds", "strict": True, "schema": LLM_SCHEMA}})
        if _PROVIDER_OPTS["openai_reasoning"] and (model.startswith("gpt-5") or model.startswith("o")):
            kwargs["reasoning_effort"] = _PROVIDER_OPTS["openai_reasoning"]   # en düşük düşünme: ucuz ve hızlı (Claude'daki effort="low" karşılığı)
        try:
            _t0 = time.perf_counter(); r = _OPENAI_CLIENT.chat.completions.create(**kwargs); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
        except Exception as e:
            if "reasoning_effort" in kwargs and ("reasoning" in str(e).lower() or "unsupported" in str(e).lower()) and "429" not in str(e):
                log(f"{model}: reasoning_effort kabul edilmedi ({str(e)[:100]}); parametresiz devam"); _PROVIDER_OPTS["openai_reasoning"] = None
                kwargs.pop("reasoning_effort"); _t0 = time.perf_counter(); r = _OPENAI_CLIENT.chat.completions.create(**kwargs); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
            else:
                raise
        _note_usage(model, getattr(r, "usage", None))
        data = extract_json(r.choices[0].message.content)
    elif provider == "google":
        if _GOOGLE_CLIENT is None:
            from google import genai
            _GOOGLE_CLIENT = genai.Client(api_key=GOOGLE_API_KEY)
        cfg = {"system_instruction": SYSTEM_PROMPT_LLM, "response_mime_type": "application/json", "response_schema": strip_additional(LLM_SCHEMA), "temperature": 0.0}
        if _PROVIDER_OPTS["google_thinking_budget"] is not None:
            cfg["thinking_config"] = {"thinking_budget": _PROVIDER_OPTS["google_thinking_budget"]}   # 0: düşünme kapalı (2.5 Flash)
        try:
            _t0 = time.perf_counter(); r = _GOOGLE_CLIENT.models.generate_content(model=model, contents=user, config=cfg); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
        except Exception as e:
            if "thinking_config" in cfg and "thinking" in str(e).lower() and "429" not in str(e):
                log(f"{model}: thinking_budget kabul edilmedi ({str(e)[:100]}); parametresiz devam"); _PROVIDER_OPTS["google_thinking_budget"] = None
                cfg.pop("thinking_config"); _t0 = time.perf_counter(); r = _GOOGLE_CLIENT.models.generate_content(model=model, contents=user, config=cfg); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
            else:
                raise
        _note_usage(model, getattr(r, "usage_metadata", None))
        data = extract_json(r.text)
    else:
        raise ValueError(provider)
    return {int(p["id"]): normalize_need(p.get("need")) for p in data.get("predictions", [])}

def llm_cache_path(provider, model, set_name):
    sig = hashlib.md5(json.dumps([set_name, GEN_SIGNATURE, LLM_EVAL_MESSAGES_PER_CALL, LLM_PROMPT_VERSION, FIX_NEED_TRANSLATIONS]).encode()).hexdigest()[:8]
    return os.path.join(DIRS["llm_cache"], f"{set_name}_{provider}_{model}_{sig}.jsonl")

def llm_proto_tag(provider, model):
    """Önbellek satırına yazılan protokol etiketi (denetim için)."""
    if provider == "anthropic":
        return f"effort={LLM_EVAL_EFFORT};thinking={thinking_param(model, THINKING_MODE)}"
    if provider == "openai":
        return f"reasoning_effort={_PROVIDER_OPTS['openai_reasoning']}"
    if provider == "google":
        return f"thinking_budget={_PROVIDER_OPTS['google_thinking_budget']};temperature=0"
    return ""

def llm_evaluate(provider, model, dfe, set_name):
    """Dönen: top1 listesi, medyan ms/mesaj, başarısız çağrı sayısı, tahminsiz mesaj sayısı, durma nedeni (None: tamamlandı)."""
    ids, texts = dfe["id"].tolist(), dfe["English"].tolist()
    cache_path = llm_cache_path(provider, model, set_name)
    cache = {}
    if os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as f:
            for line in f:
                try:
                    r = json.loads(line)
                    if r.get("need") is not None:
                        cache[int(r["id"])] = r
                except Exception:
                    pass
    todo = [(i, t) for i, t in zip(ids, texts) if i not in cache]
    batches = [todo[k:k + LLM_EVAL_MESSAGES_PER_CALL] for k in range(0, len(todo), LLM_EVAL_MESSAGES_PER_CALL)]
    if cache:
        log(f"{model} ({set_name}): {len(cache)} mesaj önbellekten, {len(todo)} mesaj ({len(batches)} istek) API'den")
    lock = threading.Lock(); n_fail = 0; n_ok = 0; stop_reason = None
    workers = LLM_EVAL_WORKERS_BY_PROVIDER.get(provider, LLM_EVAL_WORKERS) if provider != "anthropic" else LLM_EVAL_WORKERS
    proto = llm_proto_tag(provider, model)
    def work(batch):
        preds, api_ms = llm_call_paced(provider, model, batch); ms = api_ms / len(batch)   # yalnız API süresi (hız sınırı beklemesi hariç)
        return [{"id": i, "need": preds.get(i), "ms": ms, "proto": proto} for i, _ in batch]
    if batches:
        ex = ThreadPoolExecutor(max_workers=workers)
        fout = open(cache_path, "a", encoding="utf-8")
        futs = [ex.submit(work, b) for b in batches]
        bar = tqdm(total=len(futs), desc=f"BDM {model} ({set_name})")
        try:
            for fut in as_completed(futs):
                try:
                    rows = fut.result()
                except (LLMQuotaExhausted, BudgetExceeded) as e:
                    if stop_reason is None:
                        stop_reason = str(e)[:300]
                        log(f"DURDURULDU ({model}, {set_name}): {stop_reason}")
                        ex.shutdown(wait=False, cancel_futures=True)
                    n_fail += 1
                except Exception as e:
                    n_fail += 1; log(f"BDM çağrısı başarısız ({model}): {str(e)[:200]}")
                else:
                    n_ok += 1
                    with lock:
                        for r in rows:
                            if r["need"] is not None:
                                cache[r["id"]] = r
                            fout.write(json.dumps(r, ensure_ascii=False) + "\n")
                bar.update(1); bar.set_postfix({"başarılı": n_ok, "başarısız": n_fail})
                if stop_reason is not None and all(f.done() or f.cancelled() for f in futs):
                    break
        finally:
            bar.close(); fout.close(); ex.shutdown(wait=False, cancel_futures=True)
    top1 = [cache.get(i, {}).get("need") for i in ids]
    ms = [cache[i]["ms"] for i in ids if i in cache and cache[i].get("ms") is not None]
    n_missing = len(ids) - sum(p is not None for p in top1)
    return top1, (float(np.median(ms)) if ms else float("nan")), n_fail, n_missing, stop_reason

# --- Gecikme ve maliyet ölçümü: tüm BDM'ler için AYNI paylaşımlar, sıralı (tek iş parçacığı) çağrılar ---------------
def _usage_snapshot(provider, model):
    src = USAGE if provider == "anthropic" else OTHER_USAGE
    return dict(src.get(model, {}))

def _usage_delta(before, after):
    return {k: after.get(k, 0) - before.get(k, 0) for k in set(before) | set(after)}

def llm_latency_benchmark(provider, model, dfe, n_batch_req, n_single_req, force=False):
    """Toplu (LLM_EVAL_MESSAGES_PER_CALL paylaşım/istek) ve tekil (1 paylaşım/istek) kipte gecikme ve token kullanımı.
    Paylaşımlar sabit tohumla seçilir (tüm modellerde aynı). Hız sınırı beklemesi ölçüme girmez. Sonuç Drive'da önbelleğe alınır."""
    mpc = LLM_EVAL_MESSAGES_PER_CALL
    bench_dir = os.path.join(DATA_DIR, "bdm_bench"); os.makedirs(bench_dir, exist_ok=True)
    sig = hashlib.md5(json.dumps([GEN_SIGNATURE, mpc, n_batch_req, n_single_req, LLM_PROMPT_VERSION, llm_proto_tag(provider, model)]).encode()).hexdigest()[:8]
    out_path = os.path.join(bench_dir, f"{provider}_{model}_{sig}.json")
    if os.path.exists(out_path) and not force:
        return json.load(open(out_path, encoding="utf-8"))
    rng = np.random.default_rng(SEED + 2026)
    pool = rng.choice(dfe["id"].values, size=min(len(dfe), n_batch_req * mpc + n_single_req), replace=False)
    text = dict(zip(dfe["id"].tolist(), dfe["English"].tolist()))
    batch_reqs = [[(int(i), text[int(i)]) for i in pool[k * mpc:(k + 1) * mpc]] for k in range(n_batch_req)]
    single_reqs = [[(int(i), text[int(i)])] for i in pool[n_batch_req * mpc:n_batch_req * mpc + n_single_req]]
    res = {"model": model, "saglayici": provider, "protokol": llm_proto_tag(provider, model), "tamam": True, "neden": None}
    for mode, reqs in (("toplu", batch_reqs), ("tekil", single_reqs)):
        times, n_posts, n_ok = [], 0, 0
        before = _usage_snapshot(provider, model)
        for req in tqdm(reqs, desc=f"Ölçüm {model} ({mode})", leave=False):
            try:
                preds, api_ms = llm_call_paced(provider, model, req)
            except (LLMQuotaExhausted, BudgetExceeded) as e:
                res["tamam"] = False; res["neden"] = str(e)[:200]; log(f"Ölçüm durdu ({model}, {mode}): {res['neden']}"); break
            except Exception as e:
                log(f"Ölçüm çağrısı başarısız ({model}, {mode}): {str(e)[:150]}"); continue
            times.append(api_ms); n_posts += len(req); n_ok += sum(preds.get(i) is not None for i, _ in req)
        u = _usage_delta(before, _usage_snapshot(provider, model))
        cost = cost_from_counter(model, u)
        t = np.array([x for x in times if x == x], dtype=float); k = max(1, len(times))
        res[mode] = {"istek": len(times), "paylasim": n_posts, "yanitlanan": n_ok,
                     "ms_istek_medyan": float(np.median(t)) if len(t) else float("nan"), "ms_istek_p90": float(np.percentile(t, 90)) if len(t) else float("nan"),
                     "ms_paylasim_medyan": float(np.median(t)) / (mpc if mode == "toplu" else 1) if len(t) else float("nan"),
                     "giris_token_istek": u.get("input_tokens", 0) / k, "cikis_token_istek": (u.get("billed_output_tokens", u.get("output_tokens", 0))) / k,
                     "dusunme_token_istek": u.get("thinking_tokens", 0) / k,
                     "usd": cost, "usd_1000_paylasim": cost / n_posts * 1000 if n_posts else float("nan")}
        if not res["tamam"]:
            break
    if res["tamam"]:
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(res, f, ensure_ascii=False, indent=1)
    return res

def run_latency_benchmarks(models, dfe):
    rows = []
    for provider, model in models:
        key = {"anthropic": ANTHROPIC_API_KEY, "openai": OPENAI_API_KEY, "google": GOOGLE_API_KEY}.get(provider)
        if not key and not SMOKE_TEST:
            log(f"Ölçüm: {model} atlandı (anahtar yok)"); continue
        r = llm_latency_benchmark(provider, model, dfe, BENCH_BATCH_REQUESTS, BENCH_SINGLE_REQUESTS, force=BENCH_FORCE)
        row = {"Model": model, "Sağlayıcı": provider, "Protokol": r.get("protokol"), "Tamam": r.get("tamam")}
        for mode in ("toplu", "tekil"):
            m = r.get(mode, {})
            row.update({f"{mode}_istek": m.get("istek"), f"{mode}_ms_istek_medyan": m.get("ms_istek_medyan"), f"{mode}_ms_istek_p90": m.get("ms_istek_p90"),
                        f"{mode}_ms_paylasim": m.get("ms_paylasim_medyan"), f"{mode}_giris_token": m.get("giris_token_istek"),
                        f"{mode}_cikis_token": m.get("cikis_token_istek"), f"{mode}_dusunme_token": m.get("dusunme_token_istek"),
                        f"{mode}_usd_1000_paylasim": m.get("usd_1000_paylasim")})
        rows.append(row)
        def _f(x, nd=0):
            return "?" if x is None or x != x else f"{x:.{nd}f}"
        log(f"Ölçüm {model}: toplu {_f(row.get('toplu_ms_paylasim'))} ms/paylaşım (istek medyanı {_f(row.get('toplu_ms_istek_medyan'))} ms), "
            f"tekil {_f(row.get('tekil_ms_istek_medyan'))} ms, {_f(row.get('toplu_usd_1000_paylasim'), 3)} $/1.000 paylaşım" + ("" if r.get("tamam") else " (EKSİK)"))
    if rows:
        t = pd.DataFrame(rows)
        save_table(t, "tablo6b_bdm_gecikme_maliyet_olcumu")
        print(t.round(3).to_string(index=False))
    return rows

LLM_RESULTS = []; LLM_INCOMPLETE = []
for set_name, dfe_full in EVAL_SETS.items():
    dfe = dfe_full if (LLM_EVAL_MAX_MESSAGES is None or len(dfe_full) <= LLM_EVAL_MAX_MESSAGES) else dfe_full.sample(LLM_EVAL_MAX_MESSAGES, random_state=SEED).reset_index(drop=True)
    for provider, model in LLM_EVAL_MODELS:
        if provider == "openai" and not OPENAI_API_KEY:
            log(f"{model}: OPENAI_API_KEY yok, atlandı"); continue
        if provider == "google" and not GOOGLE_API_KEY:
            log(f"{model}: GOOGLE_API_KEY yok, atlandı"); continue
        tag = f"{set_name}_BDM_{model}"
        try:
            cost_before = usage_cost_usd(model) if provider == "anthropic" else float("nan")
            top1, ms_per_msg, n_fail, n_missing, stop_reason = llm_evaluate(provider, model, dfe, set_name)
            coverage = 1 - n_missing / max(1, len(dfe))
            if coverage < LLM_MIN_COVERAGE:
                LLM_INCOMPLETE.append({"veri": set_name, "saglayici": provider, "model": model, "mesaj": len(dfe), "tahminsiz": n_missing,
                                       "kapsam": round(coverage, 4), "basarisiz_cagri": n_fail, "neden": stop_reason or "başarısız çağrılar"})
                log(f"EKSİK ({tag}): {len(dfe) - n_missing}/{len(dfe)} mesaj yanıtlandı (kapsam {coverage:.1%} < {LLM_MIN_COVERAGE:.0%}); "
                    f"Tablo 5'e ALINMADI. Neden: {stop_reason or 'başarısız çağrılar'}. Sorun giderilince defteri yeniden çalıştırın: yanıtlanan mesajlar önbellekte, yalnız eksikler istenir.")
                continue
            res, hit1 = evaluate_predictions(dfe, top1, [[p] if p else [] for p in top1], tag)
            res.update({"veri": set_name, "saglayici": provider, "model": model, "ms_mesaj": ms_per_msg, "basarisiz_cagri": n_fail,
                        "tahminsiz": n_missing, "maliyet_usd": (usage_cost_usd(model) - cost_before) if provider == "anthropic" else other_usage_cost(model)})
            LLM_RESULTS.append(res); HIT1_VECTORS[tag] = hit1; DISTRICT_PREDS[tag] = district_top_pred(dfe, top1)
            save_folium_map(DISTRICT_PREDS[tag], os.path.join(DIRS["maps"], f"harita_{tag}.html"), f"{model} ({set_name})")
            log(f"{tag}: ilçe doğruluğu (makale) {res['ilce_dogruluk_makale']:.3f} | hit@1 {res['hit1']:.3f} | {ms_per_msg:.0f} ms/mesaj | tahminsiz {n_missing}"
                + (f" | {usage_summary(model)}" if provider == "anthropic" else f" | {other_usage_summary(model)} (önbellekten gelenler hariç)"))
        except Exception as e:
            log(f"HATA: {tag} değerlendirilemedi: {e}")
if LLM_INCOMPLETE:
    pd.DataFrame(LLM_INCOMPLETE).to_csv(os.path.join(DIRS["results"], "tablo5_bdm_eksik.csv"), index=False, encoding="utf-8-sig")
    log("Eksik BDM değerlendirmeleri results/tablo5_bdm_eksik.csv dosyasına yazıldı.")
if RUN_LATENCY_BENCHMARK and "sentetik" in EVAL_SETS:
    log(f"Gecikme/maliyet ölçümü: her BDM için {BENCH_BATCH_REQUESTS} toplu + {BENCH_SINGLE_REQUESTS} tekil istek (aynı paylaşımlar, sıralı)")
    run_latency_benchmarks([pm for pm in LLM_EVAL_MODELS if not any(x["model"] == pm[1] for x in LLM_INCOMPLETE)], EVAL_SETS["sentetik"])

LLM_RES = pd.DataFrame(LLM_RESULTS)
if not LLM_RES.empty:
    for set_name in EVAL_SETS:
        sub = LLM_RES[LLM_RES.veri == set_name]
        if sub.empty:
            continue
        t5 = pd.DataFrame({"Model": sub["model"], "Sağlayıcı": sub["saglayici"],
                           "İlçe doğruluğu (makale ölçütü)": [f"{fmt_tr(v)} [{fmt_tr(l)}; {fmt_tr(h)}]" for v, l, h in zip(sub.ilce_dogruluk_makale, sub.ilce_ga_alt, sub.ilce_ga_ust)],
                           "Mesaj hit@1": [f"{fmt_tr(v)} [{fmt_tr(l)}; {fmt_tr(h)}]" for v, l, h in zip(sub.hit1, sub.hit1_ga_alt, sub.hit1_ga_ust)],
                           "hit@1 (ilçe ort.)": [fmt_tr(v) for v in sub.hit1_ilce_ort], "ms/mesaj": [fmt_tr(v, 0) for v in sub.ms_mesaj],
                           "Tahminsiz": sub.tahminsiz.values, "Maliyet (USD)": [fmt_tr(v, 2) for v in sub.maliyet_usd]})
        save_table(t5, f"tablo5_bdm_{set_name}")
        sub.to_csv(os.path.join(DIRS["results"], f"tablo5_bdm_{set_name}_sayisal.csv"), index=False, encoding="utf-8-sig")
        print(f"\n[{set_name}] Tablo 5, BDM sonuçları:\n", t5.to_string(index=False))
log("BDM değerlendirmesi tamam.")

In [ ]:
# =============================================================================
# 7. ANLAMLILIK TESTLERİ VE ŞEKİLLER (Türkçe etiket, ondalık virgül, DejaVu Serif, 300 dpi)
# =============================================================================
TT_COLOR = dict(zip(TEXT_TYPES, [PALETTE[0], PALETTE[1], PALETTE[2]]))
EMB_MODELS = [m for m in ALL_MODELS if m not in BASELINE_NAMES]                 # tüm gömme modelleri (ana + ek + Türkçe)
TABLE_MODELS = [m for m in MAIN_MODELS + EXTRA_MODELS if m in MODEL_SPECS]      # üç metin türü olan modeller

def model_color(m):
    if m in MAIN_MODELS: return PALETTE[0]
    if m in BASELINE_NAMES: return PALETTE[8]
    if MODEL_SPECS.get(m, {}).get("kind") == "st_tr": return PALETTE[2]
    return PALETTE[5]

# --- Eşleştirilmiş anlamlılık testleri (aynı mesajlar, ikili sonuçlar) -------------------------------
def mcnemar_exact(a, b):
    """a, b: 0/1 isabet vektörleri. Uyuşmayan çiftler üzerinde tam binom testi (iki yönlü)."""
    n10 = int(((a == 1) & (b == 0)).sum()); n01 = int(((a == 0) & (b == 1)).sum())
    n = n10 + n01
    if n == 0:
        return 1.0, n10, n01
    try:
        from scipy.stats import binomtest
        return float(binomtest(min(n10, n01), n, 0.5, alternative="two-sided").pvalue), n10, n01
    except Exception:
        from math import comb
        k = min(n10, n01); p = sum(comb(n, i) for i in range(k + 1)) / 2 ** n
        return float(min(1.0, 2 * p)), n10, n01

def paired_bootstrap(a, b, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed); n = len(a); d = a - b
    idx = rng.integers(0, n, size=(n_boot, n))
    diffs = d[idx].mean(axis=1)
    return float(d.mean()), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

def holm(pvals):
    p = np.asarray(pvals, dtype=float); m = len(p); order = np.argsort(p); adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i]); adj[i] = min(1.0, running)
    return adj

def significance_tables(set_name):
    sub = RES[RES.veri == set_name]
    if sub.empty or not SIGNIFICANCE_TESTS:
        return
    cands = sub[sub.model.isin(EMB_MODELS)].sort_values("hit1", ascending=False)
    if cands.empty:
        return
    ref = cands.iloc[0]; ref_tag = f"{set_name}_{ref.model}_{ref.metin}"; a = HIT1_VECTORS[ref_tag]
    rows = []
    for r in sub.itertuples():
        tag = f"{set_name}_{r.model}_{r.metin}"
        if tag == ref_tag or tag not in HIT1_VECTORS or len(HIT1_VECTORS[tag]) != len(a):
            continue
        b = HIT1_VECTORS[tag]; p, n10, n01 = mcnemar_exact(a, b); d, lo, hi = paired_bootstrap(a, b)
        rows.append({"Referans": f"{ref.model} / {TEXT_TYPE_TR[ref.metin]}", "Karşılaştırılan": f"{r.model} / {TEXT_TYPE_TR[r.metin]}",
                     "hit1_ref": float(ref.hit1), "hit1_diger": float(r.hit1), "fark": d, "fark_GA_alt": lo, "fark_GA_ust": hi,
                     "sadece_ref_dogru": n10, "sadece_diger_dogru": n01, "p_McNemar": p})
    if not LLM_RES.empty:
        for r in LLM_RES[LLM_RES.veri == set_name].itertuples():
            tag = f"{set_name}_BDM_{r.model}"
            if tag in HIT1_VECTORS and len(HIT1_VECTORS[tag]) == len(a):
                b = HIT1_VECTORS[tag]; p, n10, n01 = mcnemar_exact(a, b); d, lo, hi = paired_bootstrap(a, b)
                rows.append({"Referans": f"{ref.model} / {TEXT_TYPE_TR[ref.metin]}", "Karşılaştırılan": f"{r.model} (BDM)",
                             "hit1_ref": float(ref.hit1), "hit1_diger": float(r.hit1), "fark": d, "fark_GA_alt": lo, "fark_GA_ust": hi,
                             "sadece_ref_dogru": n10, "sadece_diger_dogru": n01, "p_McNemar": p})
    if rows:
        t = pd.DataFrame(rows); t["p_Holm"] = holm(t["p_McNemar"]); t["anlamli_0.05"] = t["p_Holm"] < 0.05
        save_table(t, f"tablo8_anlamlilik_en_iyiye_karsi_{set_name}")
        t = t.copy()
        # forest plot: her model için yalnız en iyi metin türü (+ BDM ve temel çizgiler)
        t["_model"] = t["Karşılaştırılan"].str.split(" / ").str[0].str.replace(" (BDM)", "", regex=False)
        t2 = t.sort_values("hit1_diger", ascending=False).groupby("_model").head(1).sort_values("fark")
        fig, ax = plt.subplots(figsize=(7, 0.28 * len(t2) + 1.5))
        y = np.arange(len(t2))
        ax.errorbar(t2["fark"], y, xerr=[t2["fark"] - t2["fark_GA_alt"], t2["fark_GA_ust"] - t2["fark"]], fmt="o", color=PALETTE[0], ecolor="grey", capsize=2, ms=3.5)
        ax.axvline(0, color="black", lw=0.8); ax.set_yticks(y); ax.set_yticklabels(t2["Karşılaştırılan"], fontsize=7)
        ax.xaxis.set_major_formatter(TR_DECIMAL); ax.set_xlabel(f"hit@1 farkı: {ref.model} / {TEXT_TYPE_TR[ref.metin]} eksi karşılaştırılan (%95 GA)")
        ax.set_title("En iyi yapılandırmaya karşı eşleştirilmiş farklar", fontsize=10)
        fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], f"sekil_anlamlilik_{set_name}.png"), dpi=300); plt.close(fig)
    # metin türü etkisi: model içi eşleştirilmiş karşılaştırmalar
    rows = []
    for m in TABLE_MODELS:
        for t1, t2_ in (("plain", "lemma"), ("plain", "stem"), ("lemma", "stem")):
            k1, k2 = f"{set_name}_{m}_{t1}", f"{set_name}_{m}_{t2_}"
            if k1 in HIT1_VECTORS and k2 in HIT1_VECTORS and len(HIT1_VECTORS[k1]) == len(HIT1_VECTORS[k2]):
                p, n10, n01 = mcnemar_exact(HIT1_VECTORS[k1], HIT1_VECTORS[k2]); d, lo, hi = paired_bootstrap(HIT1_VECTORS[k1], HIT1_VECTORS[k2])
                rows.append({"Model": m, "Karşılaştırma": f"{TEXT_TYPE_TR[t1]} - {TEXT_TYPE_TR[t2_]}", "fark": d, "fark_GA_alt": lo, "fark_GA_ust": hi, "p_McNemar": p})
    if rows:
        t = pd.DataFrame(rows); t["p_Holm"] = holm(t["p_McNemar"]); t["anlamli_0.05"] = t["p_Holm"] < 0.05
        save_table(t, f"tablo9_metin_turu_etkisi_{set_name}")

for set_name in EVAL_SETS:
    significance_tables(set_name)

# --- Şekiller ---------------------------------------------------------------------------------
def grouped_bar(set_name, metric, lo, hi, ylabel, fname, title):
    sub = RES[RES.veri == set_name]
    models = [m for m in TABLE_MODELS if m in sub.model.values]
    if not models:
        return
    x = np.arange(len(models)); w = 0.26
    fig, ax = plt.subplots(figsize=(max(7, 1.1 * len(models) + 2), 3.8))
    for k, tt in enumerate(TEXT_TYPES):
        s = sub[sub.metin == tt].set_index("model").reindex(models)
        err = np.vstack([(s[metric] - s[lo]).clip(lower=0).fillna(0), (s[hi] - s[metric]).clip(lower=0).fillna(0)])
        ax.bar(x + (k - 1) * w, s[metric].fillna(0), w, yerr=err, capsize=2, color=TT_COLOR[tt], label=TEXT_TYPE_TR[tt], error_kw={"lw": 0.8})
    base = sub[sub.model == "Temel: Rastgele"]
    if not base.empty:
        ax.axhline(float(base[metric].iloc[0]), color="grey", ls="--", lw=0.8, label="Rastgele temel")
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=20, ha="right"); ax.set_ylim(0, 1.0)
    ax.yaxis.set_major_formatter(TR_DECIMAL); ax.set_ylabel(ylabel); ax.set_title(title, fontsize=10)
    ax.legend(frameon=False, ncol=4, loc="upper left", fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], fname), dpi=300); plt.close(fig)

def heatmap(set_name, fname):
    pv1 = TABLES.get(f"tablo4_ilce_dogrulugu_{set_name}"); pv2 = TABLES.get(f"tablo4b_hit1_{set_name}")
    if pv1 is None or pv1.empty:
        return
    pv1 = pv1.reindex([m for m in TABLE_MODELS if m in pv1.index]); pv2 = pv2.reindex(pv1.index)
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 0.45 * len(pv1) + 1.8))
    for ax, pv, title in zip(axes, (pv1, pv2), ("İlçe doğruluğu (makale ölçütü)", "Mesaj düzeyi hit@1")):
        ax.imshow(pv.values.astype(float), cmap="Blues", vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(pv.columns))); ax.set_xticklabels(pv.columns); ax.set_yticks(range(len(pv.index))); ax.set_yticklabels(pv.index)
        for i in range(pv.shape[0]):
            for j in range(pv.shape[1]):
                v = pv.values[i, j]
                if not pd.isna(v):
                    ax.text(j, i, fmt_tr(v), ha="center", va="center", fontsize=8, color="white" if v > 0.6 else "black")
        ax.set_title(title, fontsize=10)
        for s in ax.spines.values():
            s.set_visible(False)
    fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], fname), dpi=300); plt.close(fig)

def best_configs(set_name, models=None):
    sub = RES[RES.veri == set_name]
    models = [m for m in (models or ALL_MODELS) if m in sub.model.values]
    return sub.sort_values("hit1", ascending=False).groupby("model").head(1).set_index("model").reindex(models).reset_index()

def overall_fig(set_name, fname):
    best = best_configs(set_name)
    llm = LLM_RES[LLM_RES.veri == set_name] if not LLM_RES.empty else pd.DataFrame()
    names = [f"{r.model}\n({TEXT_TYPE_TR[r.metin]})" for r in best.itertuples()] + [f"{r.model}\n(BDM)" for r in llm.itertuples()]
    vals = list(best.hit1) + list(llm.hit1); los = list(best.hit1_ga_alt) + list(llm.hit1_ga_alt); his = list(best.hit1_ga_ust) + list(llm.hit1_ga_ust)
    colors = [model_color(m) for m in best.model] + [PALETTE[3]] * len(llm)
    if not names:
        return
    fig, ax = plt.subplots(figsize=(max(8, 0.85 * len(names) + 2), 4.2))
    err = np.vstack([np.clip(np.array(vals) - np.array(los), 0, None), np.clip(np.array(his) - np.array(vals), 0, None)])
    ax.bar(range(len(names)), vals, yerr=err, capsize=2, color=colors, error_kw={"lw": 0.8})
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, fontsize=6.5, rotation=30, ha="right"); ax.set_ylim(0, 1.0); ax.yaxis.set_major_formatter(TR_DECIMAL)
    ax.set_ylabel("Mesaj düzeyi hit@1")
    ax.set_title("Tüm yöntemler, en iyi metin türüyle: makale modelleri (mavi), ek (açık mavi), Türkçe (yeşil), temel çizgi (gri), BDM (turuncu)", fontsize=8.5)
    fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], fname), dpi=300); plt.close(fig)

def latency_fig(fname):
    rows = []
    sub = RES[RES.veri == "sentetik"]
    for m in EMB_MODELS:
        s = sub[sub.model == m]
        if not s.empty and float(s.ms_mesaj_tekil.median()) > 0:
            rows.append((m, float(s.ms_mesaj_tekil.median()), PALETTE[0]))
    if not LLM_RES.empty:
        for r in LLM_RES[LLM_RES.veri == "sentetik"].itertuples():
            if not pd.isna(r.ms_mesaj) and r.ms_mesaj > 0:
                rows.append((r.model, float(r.ms_mesaj), PALETTE[3]))
    if not rows:
        return
    fig, ax = plt.subplots(figsize=(max(7, 0.75 * len(rows) + 2), 3.8))
    ax.bar(range(len(rows)), [r[1] for r in rows], color=[r[2] for r in rows])
    ax.set_yscale("log"); ax.set_xticks(range(len(rows))); ax.set_xticklabels([r[0] for r in rows], rotation=30, ha="right", fontsize=7.5)
    ax.set_ylabel("Mesaj başına süre (ms, log)")
    ax.set_title("Tek mesaj gecikmesi: yerel modeller %s (mavi), BDM API'leri (turuncu)" % ("CPU" if LATENCY_ON_CPU else "GPU/CPU"), fontsize=9.5)
    for i, r in enumerate(rows):
        ax.text(i, r[1] * 1.15, fmt_tr(r[1], 1 if r[1] < 100 else 0), ha="center", fontsize=6.5)
    fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], fname), dpi=300); plt.close(fig)

def need_recall_fig(set_name, fname):
    best = best_configs(set_name, EMB_MODELS)
    if best.empty:
        return
    r = best.sort_values("hit1", ascending=False).iloc[0]; tag = f"{set_name}_{r.model}_{r.metin}"
    p = os.path.join(DIRS["preds"], f"mesaj_{tag}.csv")
    if not os.path.exists(p):
        return
    pm = pd.read_csv(p)
    rec = []
    for lab in EN_LABELS:
        mask = pm["gercek"].fillna("").str.split(";").apply(lambda L: lab in L)
        if mask.sum() >= 5:
            rec.append((EN2TR.get(lab, lab), float((pm.loc[mask, "top1"] == lab).mean()), int(mask.sum())))
    rec = sorted(rec, key=lambda x: x[2], reverse=True)[:30]
    rec = sorted(rec, key=lambda x: x[1])
    fig, ax = plt.subplots(figsize=(7, 8))
    ax.barh([f"{n} (n={c})" for n, _, c in rec], [v for _, v, _ in rec], color=PALETTE[2])
    ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(TR_DECIMAL); ax.set_xlabel("İhtiyaç bazında isabet (top-1)")
    ax.set_title(f"En iyi yapılandırma ({r.model} / {TEXT_TYPE_TR[r.metin]}): en sık 30 ihtiyaç", fontsize=10)
    fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], fname), dpi=300); plt.close(fig)
    pd.DataFrame(rec, columns=["İhtiyaç", "isabet_top1", "n"]).to_csv(os.path.join(DIRS["results"], f"ihtiyac_bazinda_isabet_{set_name}.csv"), index=False, encoding="utf-8-sig")

def explicitness_fig(fname):
    dfe = EVAL_SETS["sentetik"]
    best = best_configs("sentetik", EMB_MODELS)
    groups = ["explicit", "mixed", "implicit"]; names_tr = {"explicit": "Açık", "mixed": "Karma", "implicit": "Örtük"}
    rows = []
    for r in best.itertuples():
        tag = f"sentetik_{r.model}_{r.metin}"
        if tag in HIT1_VECTORS:
            rows.append((f"{r.model} ({TEXT_TYPE_TR[r.metin]})", HIT1_VECTORS[tag]))
    if not LLM_RES.empty:
        for r in LLM_RES[LLM_RES.veri == "sentetik"].itertuples():
            tag = f"sentetik_BDM_{r.model}"
            if tag in HIT1_VECTORS and len(HIT1_VECTORS[tag]) == len(dfe):
                rows.append((f"{r.model} (BDM)", HIT1_VECTORS[tag]))
    if not rows:
        return
    tab = pd.DataFrame({name: pd.Series(h, index=dfe.index).groupby(dfe["explicitness"]).mean().reindex(groups).values for name, h in rows}, index=[names_tr[g] for g in groups]).T
    save_table(tab.reset_index().rename(columns={"index": "Model"}), "tablo7_acik_ortuk_hit1")
    x = np.arange(len(tab)); w = 0.27
    fig, ax = plt.subplots(figsize=(max(8, 0.8 * len(tab) + 2), 4))
    for k, g in enumerate(tab.columns):
        ax.bar(x + (k - 1) * w, tab[g].values, w, color=PALETTE[k], label=g)
    ax.set_xticks(x); ax.set_xticklabels(tab.index, rotation=30, ha="right", fontsize=6.5); ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(TR_DECIMAL)
    ax.set_ylabel("hit@1"); ax.set_title("İhtiyacın açık / karma / örtük anlatıldığı paylaşımlarda isabet", fontsize=10); ax.legend(frameon=False, ncol=3)
    fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], fname), dpi=300); plt.close(fig)

for set_name in EVAL_SETS:
    grouped_bar(set_name, "ilce_dogruluk_makale", "ilce_ga_alt", "ilce_ga_ust", "İlçe doğruluğu (makale ölçütü)", f"sekil_ilce_dogrulugu_{set_name}.png", f"Model ve metin türüne göre ilçe doğruluğu ({set_name})")
    grouped_bar(set_name, "hit1", "hit1_ga_alt", "hit1_ga_ust", "Mesaj düzeyi hit@1", f"sekil_hit1_{set_name}.png", f"Model ve metin türüne göre mesaj düzeyi isabet ({set_name})")
    heatmap(set_name, f"sekil_isi_haritasi_{set_name}.png")
    overall_fig(set_name, f"sekil_genel_karsilastirma_{set_name}.png")
    need_recall_fig(set_name, f"sekil_ihtiyac_isabet_{set_name}.png")
latency_fig("sekil_gecikme.png")
explicitness_fig("sekil_acik_ortuk.png")
log("Şekiller kaydedildi:", sorted(os.listdir(DIRS["figures"])))

In [ ]:
# =============================================================================
# 8. ÖZET RAPOR (ozet.md), ORTAM BİLGİSİ, ZİP VE İNDİRME
# =============================================================================
def pkg_version(name):
    try:
        import importlib.metadata as im
        return im.version(name)
    except Exception:
        return "yok"
env_lines = [f"Tarih: {datetime.datetime.now().isoformat()}", f"Python: {sys.version.split()[0]}", f"GPU: {GPU_NAME}", f"RAM: {RAM_GB:.0f} GB",
             f"Colab: {IN_COLAB}", f"SMOKE_TEST: {SMOKE_TEST}"]
for pkg in ["anthropic", "sentence-transformers", "transformers", "torch", "gensim", "tensorflow", "tensorflow_hub", "nltk", "numpy", "pandas", "folium", "openai", "google-genai"]:
    env_lines.append(f"{pkg}: {pkg_version(pkg)}")
with open(os.path.join(DIRS["logs"], "ortam.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(env_lines))

def md_table_file(name):
    p = os.path.join(DIRS["results"], name + ".md")
    return open(p, encoding="utf-8").read() if os.path.exists(p) else "_(üretilmedi)_\n"

usage_lines = [f"- Düşünme kipi: {THINKING_MODE}; effort: üretim={GEN_EFFORT}, BDM={LLM_EVAL_EFFORT}, QC={QC_EFFORT}"]
for m in USAGE:
    u = USAGE[m]
    usage_lines.append(f"- {m}: {u.get('calls', 0)} çağrı; giriş {u.get('input_tokens', 0):,}, çıkış {u.get('output_tokens', 0):,} (düşünme {u.get('thinking_tokens', 0):,}), "
                       f"önbellek okuma {u.get('cache_read_input_tokens', 0):,}, önbellek yazma {u.get('cache_creation_input_tokens', 0):,} token; tahmini ${usage_cost_usd(m):.2f}")
usage_lines.append(f"- Oturum toplamı (tahmini): ${total_cost_usd():.2f}")

FAIRNESS_NOTES = f"""
## Eşit koşullar

Eşitlenen koşullar: (1) tüm modeller aynı mesajları ve aynı ihtiyaç listesini görür; (2) aynı ön işleme hatları (düz / lemma / gövde) ihtiyaç adlarına da uygulanır; (3) karar kuralı herkes için aynıdır (kosinüs benzerliğinde en yüksek ihtiyaç, top-1; BDM'ler için aynı listeden tek ihtiyaç); (4) hiçbir model bu veriyle eğitilmez veya ayarlanmaz (sıfır atış); (5) aynı ölçütler ve aynı bootstrap güven aralıkları; (6) tek mesaj gecikmesi tüm yerel modeller için aynı cihazda ({'CPU' if LATENCY_ON_CPU else 'karışık'}) ölçülür; (7) temel çizgiler (TF-IDF, rastgele, çoğunluk) şans düzeyini gösterir; (8) ince ayar etkisi eş çiftlerle ayrıştırılır (BERT-raw ↔ BERT-nli, ALBERT ↔ ALBERT-paraphrase); (9) en iyi modele karşı eşleştirilmiş McNemar + bootstrap fark testleri, Holm düzeltmeli (Tablo 8) ve metin türü etkisi (Tablo 9).

Eşitlenemeyen, belgelenen farklar (Tablo 3b): parametre sayısı, gömme boyutu, eğitim korpusu, alt kelime/sözlük davranışı (gövdeleme Word2Vec/GloVe'da sözlük dışı kelime üretir; FastText ve transformer'lar alt kelimelerle başa çıkar), kasa duyarlılığı. BDM'ler ihtiyaç listesini istem içinde görür ve talimat izler; gömme modelleri yalnızca vektör benzerliği kullanır. Bu farklar, modellerin "kutudan çıktığı hâliyle" karşılaştırıldığı uygulama odaklı bir çalışmada beklenen farklardır.
"""

file_list = []
for root, _, fs in os.walk(RUN_DIR):
    for fn in sorted(fs):
        file_list.append(os.path.relpath(os.path.join(root, fn), RUN_DIR))

report = [f"# Sonuç özeti: {RUN_TAG}", "", f"Üretim: {len(df_syn)} kayıt, model {GEN_MODEL}, imza {GEN_SIGNATURE}; SMOKE_TEST={SMOKE_TEST}", "",
          "## Veri özeti", md_table_file("veri_ozet"), "## Token kullanımı ve maliyet", "\n".join(usage_lines) or "_(yok)_", ""]
for set_name in EVAL_SETS:
    report += [f"## Tablo 4 ({set_name}): ilçe doğruluğu, makale ölçütü [%95 GA]", md_table_file(f"tablo4_ilce_dogrulugu_{set_name}_GA"),
               f"## Tablo 4b ({set_name}): mesaj düzeyi hit@1 [%95 GA]", md_table_file(f"tablo4b_mesaj_hit1_{set_name}_GA"),
               f"## Tablo 4c ({set_name}): mesaj düzeyi hit@3", md_table_file(f"tablo4c_mesaj_hit3_{set_name}"),
               f"## Tablo 5 ({set_name}): BDM'ler", md_table_file(f"tablo5_bdm_{set_name}")]
for set_name in EVAL_SETS:
    report += [f"## Tablo 8 ({set_name}): en iyi yapılandırmaya karşı eşleştirilmiş testler", md_table_file(f"tablo8_anlamlilik_en_iyiye_karsi_{set_name}"),
               f"## Tablo 9 ({set_name}): metin türü etkisi (model içi)", md_table_file(f"tablo9_metin_turu_etkisi_{set_name}")]
report += ["## Tablo 3b: model ayarları (eşit koşullar belgesi)", md_table_file("tablo3b_model_ayarlari"),
           "## Kalite kontrol (bağımsız model)", ("```\n" + json.dumps(qc_summary, ensure_ascii=False, indent=1) + "\n```") if qc_summary else "_(kapalı)_",
           "## Tablo 6: çıkarım süresi (ms/mesaj)", md_table_file("tablo6_cikarim_suresi"),
           "## Tablo 7: açık / karma / örtük anlatıma göre hit@1", md_table_file("tablo7_acik_ortuk_hit1"),
           "## Model kaynakları", "\n".join(f"- {k}: {v}" for k, v in MODEL_INFO.items()),
           "## Harita sınırları", (f"- {GEO_SOURCE_NOTE}; dosya: data/geo/{os.path.basename(GEOJSON_PATH)}" if GEOJSON_PATH else "- İndirilemedi; haritalar nokta işaretleriyle çizildi."),
           FAIRNESS_NOTES,
           "## Dosyalar", "\n".join(f"- {f}" for f in file_list), "", "## Ortam", "\n".join(env_lines)]
with open(os.path.join(RUN_DIR, "ozet.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(report))

# Zip: çalışma klasörü + üretilen veri seti dosyaları
zip_base = os.path.join(BASE_DIR, f"sonuclar_{RUN_TAG}")
staging = os.path.join(LOCAL_SCRATCH, f"zip_{RUN_TAG}")
shutil.rmtree(staging, ignore_errors=True)
shutil.copytree(RUN_DIR, os.path.join(staging, RUN_TAG))
os.makedirs(os.path.join(staging, RUN_TAG, "data"), exist_ok=True)
for ext in (".xlsx", ".csv", ".jsonl"):
    p = os.path.join(DATA_DIR, DATASET_BASENAME + ext)
    if os.path.exists(p):
        shutil.copy(p, os.path.join(staging, RUN_TAG, "data", os.path.basename(p)))
if GEOJSON_PATH and os.path.exists(GEOJSON_PATH):
    os.makedirs(os.path.join(staging, RUN_TAG, "data", "geo"), exist_ok=True)
    shutil.copy(GEOJSON_PATH, os.path.join(staging, RUN_TAG, "data", "geo", os.path.basename(GEOJSON_PATH)))
zip_path = shutil.make_archive(zip_base, "zip", staging)
log(f"Zip hazır: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
print("\n".join(report[:40]))
if IN_COLAB and AUTO_DOWNLOAD_ZIP:
    try:
        files.download(zip_path)
    except Exception as e:
        log("Otomatik indirme başarısız (Drive'daki zip kullanılabilir):", e)
log(f"BİTTİ. Tüm çıktılar: {RUN_DIR}")